In [43]:
import pandas as pd
import datetime
# import pymysql
import pandas.io.sql as psql
from datetime import datetime as dt
import numpy as np
import pandas.tseries.offsets as offsets
import python_ss.python_ss as ps
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import json

import os
import ast
import db_dtypes
from google.cloud import bigquery
from google.oauth2 import service_account
from google.cloud import secretmanager
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from google_auth_oauthlib.flow import InstalledAppFlow
# importでエラーが出てしまった場合は、コマンドプロンプトにて「pip install ”必要なモジュール”」でインストールしていただく必要がございます。
# 例. pip install db_dtypes

import xlsxwriter
import re

import os
from decimal import Decimal
import calendar
# import utils
# from utils import *
#importlib.reload(utils)
print(os.getcwd())



z:\Users\suehara\Documents\python\analysis\yojitu


In [44]:
pd.options.display.max_rows = 10000
pd.options.display.max_columns = 200


In [45]:
#転機IDの10000以降の手上げ情報取得
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly',
          'https://www.googleapis.com/auth/spreadsheets']
json_path = r"Z:\Users\suehara\Documents\python\analysis\yojitu\python_ss\credentials.json"

#カレンダー用の週次まとめを行う
service = ps.get_auth(SCOPES,json_path)
SPREADSHEET_ID = '11scU7ixGvt2JYSBQlHkGMKZSLSzYbrmG221CXUGDqDU'
Sheet_NAME = 'masta!'
Sheet_row = "A:D"
RANGE_NAME = Sheet_NAME+Sheet_row
master1 = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)

# ロンザンメンバーの在籍・退職異動・推移を見る
Sheet_row = "G:M"
RANGE_NAME = Sheet_NAME+Sheet_row
master2 = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)

# ヨミ表のAPソースを統一する
Sheet_row = "O:P"
RANGE_NAME = Sheet_NAME+Sheet_row
master3 = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)

# 顧客支持ポイントの掛率を統一する
Sheet_row = "R:U"
RANGE_NAME = Sheet_NAME+Sheet_row
master4 = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)

# 日付にQや同営業日の情報を加える
Sheet_NAME = 'Q営業日!'
Sheet_row = "A:O"
RANGE_NAME = Sheet_NAME+Sheet_row
Q_master = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)
Q_master['月'] = pd.to_datetime(Q_master['月'], errors='coerce')
Q_master['日付'] = pd.to_datetime(Q_master['日付'])


In [46]:
def aggregate_monthly_trends(df_transactions: pd.DataFrame, Q_master: pd.DataFrame) -> pd.DataFrame:
    """
    トランザクションデータとQマスタを結合し、Qおよび初中最終月ごとの推移を集計します。
    """
    # 1. Qマスタのメモリ最適化と不要カラムの除外
    # 結合に必要なカラムのみに絞り、結合時のメモリ消費を最小化
    df_q = Q_master[['日付', 'Q', '初中最終月']].copy()
    
    # 日付型への変換（すでに変換済みの場合は不要）
    df_q['日付'] = pd.to_datetime(df_q['日付'])
    
    # 【重要】順序付きカテゴリ型への変換
    # メモリを削減し、集計時のソート順（初月 -> 中月 -> 最終月）を保証する
    months_order = ['初月', '中月', '最終月']
    df_q['初中最終月'] = pd.Categorical(df_q['初中最終月'], categories=months_order, ordered=True)
    
    # 2. トランザクションデータの準備
    # トランザクション側の日付も datetime 型に揃える
    df_transactions['action_date'] = pd.to_datetime(df_transactions['action_date'])
    
    # 3. 結合 (Merge)
    # 日付をキーにして実績データにQと初中最終月を付与
    merged_df = pd.merge(
        df_transactions, 
        df_q, 
        left_on='action_date', 
        right_on='日付', 
        how='left'
    )
    
    # 4. 集計 (Group By)
    # observed=True を指定することで、データが存在しないカテゴリの組み合わせを無視し高速化
    # 例として 'sales' カラムや 'id' のカウント等、目的に応じて変更してください
    summary_df = (
        merged_df.groupby(['Q', '初中最終月'], observed=True)
        .agg(
            action_count=('id', 'count'),  # レコード数のカウント
            # total_sales=('sales', 'sum') # 売上合計などが必要な場合は追加
        )
        .reset_index()
    )
    
    # Q順、初中最終月順にソート（Categorical型のおかげで自然にソートされます）
    summary_df = summary_df.sort_values(['Q', '初中最終月'])
    
    return summary_df

In [47]:
#master3 = master3.rename(columns={"人マスタ.1": "人マスタ","sei_plus.1":"sei_plus"}) #カラム名変更
#master3 = master3.dropna(subset=['人マスタ', 'sei_plus'])

master1['日付'] = pd.to_datetime(master1['日付']) 

master2 = master2.dropna(subset=['人マスタ', 'sei_plus'])

master3 = master3.dropna(subset=['ヨミ表選択'])

master4 = master4.dropna(subset=['計上Q'])

In [48]:
# Bigqueryを使えるようにするためのおまじない
def access_secret_version(project_id, secret_id, version_id='latest'):
    client = secretmanager.SecretManagerServiceClient()

    name = f"projects/{project_id}/secrets/{secret_id}/versions/{version_id}"
    response = client.access_secret_version(request={"name": name})
    payload = response.payload.data.decode("UTF-8")
    return ast.literal_eval(payload)

# 上記関数を実行するコードが記載されています。こちらもそのままお使いください。
credentials = service_account.Credentials.from_service_account_info(
access_secret_version('r-group-bigdata', 'CREDENTIALS_SECRET_KEY_WORKER'),
scopes=["https://www.googleapis.com/auth/cloud-platform"],)


z:\Users\suehara\Documents\python\analysis\.venv\Lib\site-packages\google\auth\_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [49]:
# 当Qの開始日と最終日を取得する
sql = """
SELECT
  CONCAT(period,"-",quarter,"Q") AS Q,
  MIN(date) AS first_date,
  MAX(date) AS end_date
FROM `r-group-bigdata.koyomi.calendar`
GROUP BY CONCAT(period,"-",quarter,"Q")
ORDER BY first_date
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
date_df = client.query(sql).result().to_dataframe()

today = pd.to_datetime(dt.today().strftime('%Y-%m-%d'))

current_quarter_row = date_df[(date_df['first_date'] <= today) & (date_df['end_date'] >= today)]

if not current_quarter_row.empty:
    Q = current_quarter_row.iloc[0]['Q']
    first_date = current_quarter_row.iloc[0]['first_date']
    end_date = current_quarter_row.iloc[0]['end_date']
    print(f"Q: {Q}, First Date: {first_date}, End Date: {end_date}")
else:
    print("本日の日付に該当するクォーターは見つかりませんでした。")

#社員データ抽出
sql="""
select
user_id ,
sei_plus,
concat(sei,mei) as seimei
FROM `r-group-bigdata.live_company.syain`
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
syain_data = client.query(sql).result().to_dataframe()
syain_data.sample(30)

Q: 29-3Q, First Date: 2026-04-01, End Date: 2026-06-30


,user_id,sei_plus,seimei
1282,ta-akiyama,秋山た,秋山拓
3229,ma-kimura,木村真,木村真由子
1043,k-fukushima,福嶋謙,福嶋謙友
3105,shirakura,白倉健,白倉健翔
3891,miko-sato,佐藤未２,佐藤未琴
158,nyugaku,入学琴,入学琴子
93,iwakiri,iwakiri,岩切萌子
336,nara,奈良知,奈良知実
3003,ay-sato,佐藤文,佐藤文香
5008,m-oota,太田み,太田みどり


## 初期交渉データ

In [50]:
#初期交渉のDBから基本データを抽出（初期交渉データ）
#AP獲得者がいる場合は、AP担当はAP獲得者。
#AP獲得者が空欄でAPソース：パートナー紹介の場合はAP担当は紹介受領者。
#AP獲得者が空欄でAPソース：人事部、転機は面談担当者。
#AP獲得者が空欄でAPソースがパートナー紹介、人事部、転機の場合もAP担当は面談担当者。
#且つAP獲得者がロンザン所属でない場合（所属フラグ空欄）は面談担当者にする。

sql = """
WITH
-- CTE 1: APソースのマスタデータを準備
ap_source_master AS (
  SELECT
    code,
    name
  FROM `r-group-bigdata.live_rhs.sys_consts`
  WHERE group_code = 19),

-- CTE 2: 役職レイヤーのマスタデータを準備
layer_master AS (
  SELECT
    code,
    name
  FROM `r-group-bigdata.live_rhs.sys_consts`
  WHERE group_code = 25),

-- ※CTE 3 (first_interview_person) は不要になったため削除しました

-- CTE 3: メインとなる交渉データに必要な情報を付与し、基本的な変換処理を行う
base_data AS (
  SELECT
    shoki.id,
    shoki.tenki_id,
    shoki.kohosha_id,
    CASE
      WHEN consts.name = '転機社長名鑑' THEN '転機'
      WHEN consts.name = '人事部経由' THEN '人事部'
      WHEN consts.name IN ('顧問名鑑登録　解放者', '社外取締役名鑑　候補者') THEN '解放顧問'
      WHEN consts.name IN ('HP反響', '上場企業役員DM', 'Gアポ') THEN 'その他'
      ELSE consts.name END AS APsource,
    layer.name AS layer,
    shoki.annual_income,
    COALESCE(syi1.sei_plus, shoki.mendan_tanto) AS mendan_tanto,
    syi2.sei_plus AS ap_kakutokusha,
    shoki.kosho_setteibi,
    shoki.kosho_yoteibi,
    shoki.kosho_jisshibi,
    shoki.tsr_code,
    shoki.kosho_seq,
    shoki.saikosho_kaisu,
    shoki.saikosho_seq,
    shoki.valid_flag,
    -- ウィンドウ関数を使い、候補者ごとに前回交渉実施日からの経過日数を計算
    DATE_DIFF(
      shoki.kosho_setteibi,
      LAG(shoki.kosho_jisshibi) OVER (PARTITION BY shoki.kohosha_id ORDER BY shoki.kosho_setteibi),
      DAY) AS keikabi
  FROM `r-group-bigdata.live_rhs.shokikoshos` AS shoki
  LEFT JOIN `r-group-bigdata.live_rhs.kohoshas` AS khs ON shoki.kohosha_id = khs.id
  LEFT JOIN `r-group-bigdata.live_company.syain` AS syi1 ON shoki.mendan_tanto = syi1.user_id
  LEFT JOIN `r-group-bigdata.live_company.syain` AS syi2 ON shoki.ap_kakutoku = syi2.user_id
  -- first_interview_person の JOIN は削除しました
  LEFT JOIN ap_source_master AS consts ON shoki.ap_source = consts.code
  LEFT JOIN layer_master AS layer ON shoki.max_bushoyakushoku = layer.code),

-- CTE 4: 1つ前のAPsourceの値を取得
data_with_prev_apsource AS (
  SELECT
    *,
    -- ウィンドウ関数を使い、1つ前のAPsourceを取得
    LAG(APsource) OVER (PARTITION BY kohosha_id ORDER BY kosho_setteibi) AS prev_APsource
  FROM base_data),

-- CTE 5: APsourceが変更されたかどうかのフラグを計算
data_with_aps_change AS (
  SELECT
    *,
    -- APsourceが前回から変更された場合に1を立てる
    CASE
      WHEN APsource != prev_APsource THEN 1
      ELSE 0 END AS APS_change
  FROM data_with_prev_apsource),

-- 新設 CTE 6: first_mendan_tantoを計算する前に、まずここで sai_flg を確定させる
calc_sai_flg AS (
  SELECT
    *,
    CASE
      WHEN kosho_seq = 1 THEN 1 -- 初回交渉
      WHEN APS_change = 1 THEN 1 -- APsourceが変更された場合
      WHEN keikabi > 90 THEN 2 -- 前回実施から90日以上経過
      ELSE 0
    END AS sai_flg
  FROM data_with_aps_change)

-- 最終的なSELECT文
SELECT
  id,
  tenki_id,
  kohosha_id,
  APsource,
  layer,
  annual_income,
  mendan_tanto as kohosha_tanto,
  ap_kakutokusha,
  kosho_setteibi,
  kosho_yoteibi,
  kosho_jisshibi,
  kosho_seq,
  APS_change,
  keikabi,
  sai_flg,
  
  -- ★新しいロジック★
  -- 直近の `sai_flg = 1` になった時の `mendan_tanto` を取得して引き継ぐ
  LAST_VALUE(CASE WHEN sai_flg = 1 THEN mendan_tanto END IGNORE NULLS) OVER (
    PARTITION BY kohosha_id 
    ORDER BY kosho_setteibi 
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS first_mendan_tanto,
  
  tsr_code
FROM calc_sai_flg
# where kohosha_id = 38423
ORDER BY kosho_setteibi ASC;
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
shokikosho_data = client.query(sql).result().to_dataframe()


In [51]:
# ==========================================
# セル1: データ型の変換（より高速な一括処理）
# ==========================================
# 1. 'dbdate' 型のカラムを自動で探して、applyを使って一括で日付型に変換します
dbdate_columns = shokikosho_data.select_dtypes(include=['dbdate']).columns
shokikosho_data[dbdate_columns] = shokikosho_data[dbdate_columns].apply(pd.to_datetime)

# 2. 日付型「以外」のカラムをすべて 'object' 型に変換します
columns_to_convert = shokikosho_data.select_dtypes(exclude=['datetime', 'datetime64']).columns
shokikosho_data[columns_to_convert] = shokikosho_data[columns_to_convert].astype('object')

# 3. 確認
print(shokikosho_data.dtypes)

id                            object
tenki_id                      object
kohosha_id                    object
APsource                      object
layer                         object
annual_income                 object
kohosha_tanto                 object
ap_kakutokusha                object
kosho_setteibi        datetime64[ns]
kosho_yoteibi         datetime64[ns]
kosho_jisshibi        datetime64[ns]
kosho_seq                     object
APS_change                    object
keikabi                       object
sai_flg                       object
first_mendan_tanto            object
tsr_code                      object
dtype: object


In [52]:
# ==========================================
# セル2: Q_masterのマージ（メソッドチェーンでスッキリと）
# ==========================================
q_master_subset = Q_master[['日付', 'Q', '月', '営業日', '同営業日比較']]

# 設定日のマージとリネームを1つの処理で繋げて行います
shokikosho_data = (
    shokikosho_data.merge(q_master_subset, left_on='kosho_setteibi', right_on='日付', how='left')
    .drop(columns=['日付'])
    .rename(columns={'Q': 'setteibi_Q', '月': 'setteibi_月', '営業日': 'setteibi_営業日', '同営業日比較': 'setteibi_同営業日比較'})
)

# 実施日のマージとリネームも同様に行います
shokikosho_data = (
    shokikosho_data.merge(q_master_subset, left_on='kosho_jisshibi', right_on='日付', how='left')
    .drop(columns=['日付'])
    .rename(columns={'Q': 'jisshibi_Q', '月': 'jisshibi_月', '営業日': 'jisshibi_営業日', '同営業日比較': 'jisshibi_同営業日比較'})
)

In [53]:
# ==========================================
# セル3: master2（社員マスタ）のマージ
# ==========================================
master2_subset = master2[['Q', 'sei_plus', '職種', 'ロンザン所属フラグ', 'レイヤー']]

# 面談担当者情報のマージ
shokikosho_data = (
    shokikosho_data.merge(master2_subset, left_on=['setteibi_Q', 'kohosha_tanto'], right_on=['Q', 'sei_plus'], how='left')
    .drop(columns=['Q', 'sei_plus'])
    .rename(columns={'職種': '候担_職種', 'ロンザン所属フラグ': '候担_ロンザン所属フラグ', 'レイヤー': '候担_レイヤー'})
)

In [54]:
# ==========================================
# セル4: データの抽出、整形、縦積み（劇的にコードを削減）
# ==========================================
# 1. 対象データ(sai_flg == 1)を一度だけ抽出します
base_shoki = shokikosho_data[shokikosho_data['sai_flg'] == 1].copy()

# 2. 共通で引き継ぐカラム名と、新しく付与するキレイなカラム名のリストを準備します
common_cols = ['id', 'kohosha_id', 'APsource', 'kohosha_tanto', '候担_職種', '候担_ロンザン所属フラグ', '候担_レイヤー']
clean_cols = ['Q', '月', '営業日', '同営業日', '日付', 'KPI_ID', '候補者ID', 'APソース', '担当', '職種', '候担_ロンザン所属フラグ', 'レイヤー']

# 3. 設定データを作成（抽出と一括リネーム）
df_settei = base_shoki[['setteibi_Q', 'setteibi_月', 'setteibi_営業日', 'setteibi_同営業日比較', 'kosho_setteibi'] + common_cols].copy()
df_settei.columns = clean_cols  # カラム名を一気に上書き！
df_settei['type'] = 'shoki_settei'

# 4. 実施データを作成（抽出と一括リネーム）
df_jisshi = base_shoki[['jisshibi_Q', 'jisshibi_月', 'jisshibi_営業日', 'jisshibi_同営業日比較', 'kosho_jisshibi'] + common_cols].copy()
df_jisshi.columns = clean_cols  # カラム名を一気に上書き！
df_jisshi['type'] = 'shoki_jisshi'

# 5. データを縦に繋げ、共通の値(value)を入れます
shoki_combined_data = pd.concat([df_settei, df_jisshi], ignore_index=True)
shoki_combined_data['value'] = 1

# 結合後のデータを確認
print("結合後のデータの行数と列数:", shoki_combined_data.shape)
display(shoki_combined_data.head())


結合後のデータの行数と列数: (154320, 14)


,Q,月,営業日,同営業日,日付,KPI_ID,候補者ID,APソース,担当,職種,候担_ロンザン所属フラグ,レイヤー,type,value
0,NaN,NaT,NaN,NaN,NaT,90208,74564,SMAP,伊藤大２,NaN,NaN,NaN,shoki_settei,1
1,18-1Q,2014-11-01,26,同営業日,2014-11-20,637,1742,解放顧問,隆郁,NaN,NaN,NaN,shoki_settei,1
2,18-1Q,2014-12-01,32,同営業日,2014-12-01,2237,3613,パートナー紹介,角田隆,NaN,NaN,NaN,shoki_settei,1
3,18-2Q,2015-01-01,19,同営業日,2015-01-30,1135,2418,SMAP,隆郁,NaN,NaN,NaN,shoki_settei,1
4,18-2Q,2015-02-01,30,同営業日,2015-02-17,1975,3260,SMAP,隆郁,NaN,NaN,NaN,shoki_settei,1


In [55]:
# ==========================================
# 1. あなたが並べたい理想の順番を「リスト」で定義します
# 今後KPIが増えたら、ここにどんどんカンマ区切りで追記していくだけでOKです！
# ==========================================
type_order = [
    'shoki_settei',
    'shoki_jisshi'
]

# ==========================================
# 2. shoki_combined_data の 'type' カラムに、上で作った「独自の順番（ルール）」を記憶させます
# ==========================================
shoki_combined_data['type'] = pd.Categorical(
    shoki_combined_data['type'], 
    categories=type_order, 
    ordered=True  # 「この順番に意味があるよ（順序付きだよ）」と教えてあげます
)

In [56]:
# ==========================================
# セル5: ピボットテーブルの作成
# ==========================================
def aggregate_phase_data(df: pd.DataFrame, valid_members: list, type_order: list, 
                         agg_configs: list, agg_val: str, agg_func: str) -> pd.DataFrame:
    """
    いろいろな集計（Q、月、同じ営業日）を一度にやって、横につなげる「魔法の箱」です。
    """
    df = df.copy(deep=False) 
    df['Q_同営業日'] = df['Q'].astype(str) + "同営業日"
    
    time_columns = ['Q', '月', 'Q_同営業日']
    time_axis_dfs = []
    
    for time_col in time_columns:
        axis_dfs = []
        for col, axis_name, need_filter in agg_configs:
            # 1つずつの項目（APソースなど）を箱に分ける
            pivot = df.pivot_table(
                index=["type", col], columns=time_col,
                aggfunc=agg_func, values=agg_val
            ).fillna(0).reset_index()
            
            if need_filter:
                pivot = pivot[pivot[col].isin(valid_members)]
                
            pivot = pivot.rename(columns={col: '項目名'})
            pivot.insert(1, '集計軸', axis_name)
            axis_dfs.append(pivot)
            
        # 縦につなげる
        combined_axis = pd.concat(axis_dfs, ignore_index=True)
        combined_axis = combined_axis.set_index(["type", "集計軸", "項目名"])
        time_axis_dfs.append(combined_axis)

    # 横につなげて完成！
    final_df = pd.concat(time_axis_dfs, axis=1).fillna(0).reset_index()
    final_df['type'] = pd.Categorical(final_df['type'], categories=type_order, ordered=True)
    
    numeric_cols = final_df.columns.difference(['type', '集計軸', '項目名'])
    final_df = final_df[final_df[numeric_cols].sum(axis=1) > 0]
    
    return final_df.sort_values(by=['type', '集計軸', '項目名']).reset_index(drop=True)

# --- 実行部分 ---
valid_members = master2['sei_plus'].dropna().unique()

# ★ここを追加：全データをまとめるためのダミー列
shoki_combined_data['全体'] = '合計'

# 【初期交渉】用の設定
shoki_configs = [
    ("全体", "全体", False), # ★ここを追加
    ("APソース", "APソース", False),
    ("職種", "職種", False),
    ("レイヤー", "レイヤー", False),
    ("担当", "担当", True)
]

# 初期交渉の集計を実行（aggfunc="count", values="value"）
shoki_pivot_df = aggregate_phase_data(
    shoki_combined_data, valid_members, type_order, 
    shoki_configs, agg_val="value", agg_func="count"
)

# 確認
print("【初期交渉の新しい集計テーブル】")
display(shoki_pivot_df.head())

C:\Users\suehara\AppData\Local\Temp\ipykernel_22204\2657028705.py:19: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot = df.pivot_table(
C:\Users\suehara\AppData\Local\Temp\ipykernel_22204\2657028705.py:19: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot = df.pivot_table(
C:\Users\suehara\AppData\Local\Temp\ipykernel_22204\2657028705.py:19: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot = df.pivot_table(
C:\Users\suehara\AppData\Local\Temp\ipykernel_22204\2657028705.py:19: FutureWarning: The

【初期交渉の新しい集計テーブル】


,type,集計軸,項目名,18-1Q,18-2Q,18-3Q,18-4Q,19-1Q,19-2Q,19-3Q,19-4Q,20-1Q,20-2Q,20-3Q,20-4Q,21-1Q,21-2Q,21-3Q,21-4Q,22-1Q,22-2Q,22-3Q,22-4Q,23-1Q,23-2Q,23-3Q,23-4Q,24-1Q,24-2Q,24-3Q,24-4Q,25-1Q,25-2Q,25-3Q,25-4Q,26-1Q,26-2Q,26-3Q,26-4Q,27-1Q,27-2Q,27-3Q,27-4Q,28-1Q,28-2Q,28-3Q,28-4Q,29-1Q,29-2Q,29-3Q,2014-11-01 00:00:00,2014-12-01 00:00:00,2015-01-01 00:00:00,2015-02-01 00:00:00,2015-03-01 00:00:00,2015-04-01 00:00:00,2015-07-01 00:00:00,2015-10-01 00:00:00,2015-11-01 00:00:00,2015-12-01 00:00:00,2016-01-01 00:00:00,2016-02-01 00:00:00,2016-03-01 00:00:00,2016-04-01 00:00:00,2016-05-01 00:00:00,2016-06-01 00:00:00,2016-07-01 00:00:00,2016-08-01 00:00:00,2016-09-01 00:00:00,2016-10-01 00:00:00,2016-11-01 00:00:00,2016-12-01 00:00:00,2017-01-01 00:00:00,2017-02-01 00:00:00,2017-03-01 00:00:00,2017-04-01 00:00:00,2017-05-01 00:00:00,2017-06-01 00:00:00,2017-07-01 00:00:00,2017-08-01 00:00:00,2017-09-01 00:00:00,2017-10-01 00:00:00,2017-11-01 00:00:00,2017-12-01 00:00:00,2018-01-01 00:00:00,2018-02-01 00:00:00,2018-03-01 00:00:00,2018-04-01 00:00:00,2018-05-01 00:00:00,2018-06-01 00:00:00,2018-07-01 00:00:00,2018-08-01 00:00:00,2018-09-01 00:00:00,2018-10-01 00:00:00,2018-11-01 00:00:00,2018-12-01 00:00:00,2019-01-01 00:00:00,2019-02-01 00:00:00,2019-03-01 00:00:00,2019-04-01 00:00:00,...,2022-03-01 00:00:00,2022-04-01 00:00:00,2022-05-01 00:00:00,2022-06-01 00:00:00,2022-07-01 00:00:00,2022-08-01 00:00:00,2022-09-01 00:00:00,2022-10-01 00:00:00,2022-11-01 00:00:00,2022-12-01 00:00:00,2023-01-01 00:00:00,2023-02-01 00:00:00,2023-03-01 00:00:00,2023-04-01 00:00:00,2023-05-01 00:00:00,2023-06-01 00:00:00,2023-07-01 00:00:00,2023-08-01 00:00:00,2023-09-01 00:00:00,2023-10-01 00:00:00,2023-11-01 00:00:00,2023-12-01 00:00:00,2024-01-01 00:00:00,2024-02-01 00:00:00,2024-03-01 00:00:00,2024-04-01 00:00:00,2024-05-01 00:00:00,2024-06-01 00:00:00,2024-07-01 00:00:00,2024-08-01 00:00:00,2024-09-01 00:00:00,2024-10-01 00:00:00,2024-11-01 00:00:00,2024-12-01 00:00:00,2025-01-01 00:00:00,2025-02-01 00:00:00,2025-03-01 00:00:00,2025-04-01 00:00:00,2025-05-01 00:00:00,2025-06-01 00:00:00,2025-07-01 00:00:00,2025-08-01 00:00:00,2025-09-01 00:00:00,2025-10-01 00:00:00,2025-11-01 00:00:00,2025-12-01 00:00:00,2026-01-01 00:00:00,2026-02-01 00:00:00,2026-03-01 00:00:00,2026-04-01 00:00:00,2026-05-01 00:00:00,2026-06-01 00:00:00,18-1Q同営業日,18-2Q同営業日,18-3Q同営業日,18-4Q同営業日,19-1Q同営業日,19-2Q同営業日,19-3Q同営業日,19-4Q同営業日,20-1Q同営業日,20-2Q同営業日,20-3Q同営業日,20-4Q同営業日,21-1Q同営業日,21-2Q同営業日,21-3Q同営業日,21-4Q同営業日,22-1Q同営業日,22-2Q同営業日,22-3Q同営業日,22-4Q同営業日,23-1Q同営業日,23-2Q同営業日,23-3Q同営業日,23-4Q同営業日,24-1Q同営業日,24-2Q同営業日,24-3Q同営業日,24-4Q同営業日,25-1Q同営業日,25-2Q同営業日,25-3Q同営業日,25-4Q同営業日,26-1Q同営業日,26-2Q同営業日,26-3Q同営業日,26-4Q同営業日,27-1Q同営業日,27-2Q同営業日,27-3Q同営業日,27-4Q同営業日,28-1Q同営業日,28-2Q同営業日,28-3Q同営業日,28-4Q同営業日,29-1Q同営業日,29-2Q同営業日,29-3Q同営業日,nan同営業日
0,shoki_settei,APソース,SMAP,0,2,0,0,0,10,43,17,36,58,40,41,36,50,42,76,71,97,112,101,106,96,180,144,231,265,185,334,183,184,203,508,876,1173,928,930,1128,1192,1282,1414,1429,1427,1545,1568,1622,1577,1042,0,0,1,1,0,0,0,0,0,0,6,2,2,19,9,16,9,7,1,8,20,8,17,18,22,16,14,10,15,14,12,12,18,6,16,14,20,3,7,32,30,25,20,22,32,18,27,29,43,32,...,74,64,69,70,115,160,230,273,320,297,258,458,467,250,357,311,276,337,290,366,398,402,398,401,398,383,454,439,508,350,532,490,494,470,490,439,516,508,529,467,628,428,510,504,540,597,467,494,597,487,414,141,0,2,0,0,0,10,43,17,36,58,40,41,36,50,42,76,71,97,112,101,106,96,180,144,231,265,185,334,183,184,203,508,876,1173,928,930,1128,1192,1282,1414,1429,1427,1545,1568,1622,1577,1042,1
1,shoki_settei,APソース,その他,0,0,0,1,1,11,26,30,57,118,68,142,77,132,115,106,76,64,61,72,65,72,83,72,96,114,87,59,64,64,48,66,54,27,45,44,62,85,82,93,79,95,105,132,84,91,52,0,0,0,0,0,0,1,0,0,1,3,2,6,5,11,10,21,6,3,22,24,11,38,41,39,27,19,23,35,28,78,29,27,21,46,49,37,37,47,31,48,32,24,32,20,27,21,24,18,20,...,20,13,21,13,29,27,10,19,20,18,20,2,2,13,9,23,15,15,13,20,27,16,27,26,32,32,31,20,23,26,43,28,29,23,37,27,33,36,36,32,38,41,51,29,25,31,31,36,23,23,25,4,0,0,0,1

## 本交渉データ

In [ ]:
sql = """
WITH
-- =================================================================
-- マスタデータ準備セクション
-- 必要なマスタデータを事前にCTEとして定義し、再利用しやすくする
-- =================================================================

-- CTE 1: APソースのマスタデータ (group_code = 19)
ap_source_master AS (
  SELECT code, name FROM `r-group-bigdata.live_rhs.sys_consts` WHERE group_code = 19
),

-- CTE 2: 役職レイヤーのマスタデータ (group_code = 25)
layer_master AS (
  SELECT code, name FROM `r-group-bigdata.live_rhs.sys_consts` WHERE group_code = 25
),

-- CTE 3: ヨミのマスタデータ (group_code = 9)
yomi_master AS (
  SELECT code, name FROM `r-group-bigdata.live_rhs.sys_consts` WHERE group_code = 9
),

-- CTE 4: 役職クラスのマスタデータ (group_code = 10)
yakushoku_class_master AS (
  SELECT code, name FROM `r-group-bigdata.live_rhs.sys_consts` WHERE group_code = 10
),

-- =================================================================
-- 初期交渉データ準備セクション (元の`SHOKIS` CTEに相当)
-- =================================================================

-- CTE 5: 候補者ごとの初回接触日を計算
shokikoshos_with_initial_date AS (
  SELECT
    *,
    FIRST_VALUE(kosho_jisshibi IGNORE NULLS) OVER (PARTITION BY kohosha_id ORDER BY kosho_jisshibi) AS initial_contact_date
  FROM
    `r-group-bigdata.live_rhs.shokikoshos`
),

-- CTE 6: 初期交渉データの中間処理 (フラグ計算の前段階)
shoki_base AS (
  SELECT
    shk.id,
    shk.tenki_id,
    shk.kohosha_id,
    shk.ap_source,
    layer.name AS layer,
    shk.annual_income,
    COALESCE(syi1.sei_plus, shk.mendan_tanto) AS mendan_tanto,
    syi2.sei_plus AS ap_kakutokusha,
    shk.kosho_setteibi,
    shk.kosho_yoteibi,
    shk.kosho_jisshibi,
    shk.kosho_seq,
    shk.saikosho_kaisu,
    shk.saikosho_seq,
    shk.valid_flag,
    shk.initial_contact_date,
    -- 前回交渉実施日からの経過日数を計算
    DATE_DIFF(shk.kosho_setteibi, LAG(shk.kosho_jisshibi) OVER (PARTITION BY shk.kohosha_id ORDER BY shk.kosho_jisshibi), DAY) AS keikabi
  FROM
    shokikoshos_with_initial_date AS shk
    LEFT JOIN `r-group-bigdata.live_company.syain` AS syi1 ON shk.mendan_tanto = syi1.user_id
    LEFT JOIN `r-group-bigdata.live_company.syain` AS syi2 ON shk.ap_kakutoku = syi2.user_id
    LEFT JOIN layer_master AS layer ON shk.max_bushoyakushoku = layer.code
),

-- CTE 7: 初期交渉データの sai_flg を計算
shoki_calc_sai_flg AS (
  SELECT
    *,
    -- sai_flgを計算
    CASE
      WHEN kosho_seq = 1 THEN 1 -- 初回交渉
      -- APソースが前回から変更された場合
      WHEN ap_source != LAG(ap_source) OVER (PARTITION BY kohosha_id ORDER BY kosho_setteibi) THEN 1
      WHEN keikabi > 90 THEN 2 -- 前回実施から90日以上経過
      ELSE 0
    END AS sai_flg
  FROM
    shoki_base
),

-- 新設 CTE 7.5: 初期交渉データ内で first_mendan_tanto を引き継ぎ計算する
shoki_final AS (
  SELECT
    *,
    -- 直近の `sai_flg = 1` になった時の `mendan_tanto` を取得して引き継ぐ
    LAST_VALUE(CASE WHEN sai_flg = 1 THEN mendan_tanto END IGNORE NULLS) OVER (
      PARTITION BY kohosha_id 
      ORDER BY kosho_setteibi 
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS first_mendan_tanto
  FROM
    shoki_calc_sai_flg
),

-- =================================================================
-- 本交渉データ準備セクション (元の`HONS` CTEに相当)
-- =================================================================

-- CTE 8: 案件ごとの最新のヨミを取得
latest_yomis AS (
  SELECT
    anken_id,
    yomi
  FROM (
    SELECT
      anken_id,
      yomi,
      ROW_NUMBER() OVER (PARTITION BY anken_id ORDER BY yomi_torokubi DESC) AS rn
    FROM
      `r-group-bigdata.live_rhs.honkosho_yomis`
  )
  WHERE
    rn = 1
),

-- CTE 10: 本交渉データと関連データを結合
hons_base AS (
  SELECT
    hon.id AS honkosho_id,
    hon.anken_id,
    an.kohosha_id,
    an.linked_shokikosho_id,
    kgy.tsr_code,
    kgy.name AS kigyo_name,
    -- APソースを初期交渉と候補者マスタから取得し、優先度付け
    COALESCE(consts1.name, consts2.name) AS AP_source_raw,
    shoki.annual_income,
    shoki.layer,
    shoki.initial_contact_date,
    shoki.kosho_setteibi AS shoki_setteibi,
    shoki.kosho_jisshibi AS shoki_jisshibi,
    hon.kosho_setteibi AS hon_setteibi,
    hon.kosho_yoteibi AS hon_yoteibi,
    hon.kosho_jisshibi AS hon_jisshibi,
    COALESCE(syi1.sei_plus, hon.kohosha_tanto) AS kohosha_tanto,
    syi2.sei_plus AS kigyo_tanto,
    hon.kosho_seq AS hon_seq,
    shoki.kosho_seq AS shoki_seq,
    shoki.sai_flg,
    DATE_DIFF(hon.kosho_setteibi, shoki.kosho_jisshibi, DAY) AS jisshibi_sa,
    yomi1.name AS yomi,
    -- 最終的なヨミを決定
    COALESCE(yomi2.name, yomi1.name) AS yomi_final,
    ROUND(tsr.tokikessan_uriagedaka / 100000, 0) AS uriage,
    consts3.name AS yakushoku_class,
    -- shoki_final で計算した first_mendan_tanto を取得
    shoki.first_mendan_tanto,
        case when an.hanjokin = 20 then "半常勤"
         else "" end as hanjokin
  FROM
    `r-group-bigdata.live_rhs.honkoshos` AS hon
    LEFT JOIN `r-group-bigdata.live_rhs.ankens` AS an ON hon.anken_id = an.id
    LEFT JOIN shoki_final AS shoki ON an.linked_shokikosho_id = shoki.id
    LEFT JOIN `r-group-bigdata.live_rhs.kigyos` AS kgy ON an.kigyo_id = kgy.id
    LEFT JOIN `r-group-bigdata.live_rhs.kohoshas` AS koho ON an.kohosha_id = koho.id
    LEFT JOIN `r-group-bigdata.tsr.company_info` AS tsr ON kgy.tsr_code = tsr.tsr_code
    LEFT JOIN `r-group-bigdata.live_company.syain` AS syi1 ON hon.kohosha_tanto = syi1.user_id
    LEFT JOIN `r-group-bigdata.live_company.syain` AS syi2 ON an.kigyo_tanto = syi2.user_id
    LEFT JOIN latest_yomis AS ly ON an.id = ly.anken_id
    -- マスタ結合
    LEFT JOIN ap_source_master AS consts1 ON shoki.ap_source = consts1.code
    LEFT JOIN ap_source_master AS consts2 ON koho.ap_source = consts2.code
    LEFT JOIN yomi_master AS yomi1 ON hon.yomi = yomi1.code
    LEFT JOIN yomi_master AS yomi2 ON ly.yomi = yomi2.code
    LEFT JOIN yakushoku_class_master AS consts3 ON hon.yakushoku_class = consts3.code
  WHERE hon.kosho_seq = 1
)

-- =================================================================
-- 最終的な出力
-- =================================================================
SELECT
  honkosho_id as id,
  anken_id,
  kohosha_id,
  linked_shokikosho_id,
  tsr_code,
  kigyo_name,
  annual_income,
  layer,
  initial_contact_date,
  
  -- ★修正ポイント★ FORMAT_DATE（文字化）を外し、CASTで純粋な日付型（DATE）として出力します
  CAST(shoki_setteibi AS DATE) AS shoki_setteibi,
  CAST(shoki_jisshibi AS DATE) AS shoki_jisshibi,
  CAST(hon_setteibi AS DATE) AS hon_setteibi,
  CAST(hon_yoteibi AS DATE) AS hon_yoteibi,
  CAST(hon_jisshibi AS DATE) AS hon_jisshibi,
  
  kohosha_tanto,
  kigyo_tanto,
  hon_seq,
  shoki_seq,
  jisshibi_sa,
  yomi,
  yomi_final,
  uriage,
  yakushoku_class,
  first_mendan_tanto,
  hanjokin,
  -- APsourceを分かりやすいカテゴリに分類
  CASE
    WHEN AP_source_raw = '転機社長名鑑' THEN '転機'
    WHEN AP_source_raw = '人事部経由' THEN '人事部紹介'
    WHEN AP_source_raw IN ('顧問名鑑登録　解放者', '社外取締役名鑑　候補者') THEN '顧問名鑑登録者'
    WHEN AP_source_raw IN ('HP反響', '上場企業役員DM', 'Gアポ') THEN 'その他'
    ELSE AP_source_raw
  END AS APsource,
  -- 最終的なフラグ `sai_flg2` を計算
  CASE
    WHEN shoki_jisshibi IS NULL THEN 2
    WHEN jisshibi_sa > 90 THEN 2
    ELSE sai_flg
  END AS sai_flg2
FROM hons_base
ORDER BY
  anken_id;
"""

client = bigquery.Client(credentials=credentials, project=credentials.project_id)
honkosho_data = client.query(sql).result().to_dataframe()

In [ ]:
honkosho_data.dtypes

id                        Int64
anken_id                  Int64
kohosha_id                Int64
linked_shokikosho_id      Int64
tsr_code                 object
kigyo_name               object
annual_income           float64
layer                    object
initial_contact_date     dbdate
shoki_setteibi           dbdate
shoki_jisshibi           dbdate
hon_setteibi             dbdate
hon_yoteibi              dbdate
hon_jisshibi             dbdate
kohosha_tanto            object
kigyo_tanto              object
hon_seq                   Int64
shoki_seq                 Int64
jisshibi_sa               Int64
yomi                     object
yomi_final               object
uriage                  float64
yakushoku_class          object
first_mendan_tanto       object
hanjokin                 object
APsource                 object
sai_flg2                  Int64
dtype: object

In [ ]:
# ==========================================
# セル1: データ型の変換（より高速な一括処理）
# ==========================================
# 1. 'dbdate' 型のカラムを自動で探して、applyを使って一括で日付型に変換します
dbdate_columns = honkosho_data.select_dtypes(include=['dbdate']).columns
honkosho_data[dbdate_columns] = honkosho_data[dbdate_columns].apply(pd.to_datetime)

# 2. 日付型「以外」のカラムをすべて 'object' 型に変換します
columns_to_convert = honkosho_data.select_dtypes(exclude=['datetime', 'datetime64']).columns
honkosho_data[columns_to_convert] = honkosho_data[columns_to_convert].astype('object')

# 3. 確認
print(honkosho_data.dtypes)

id                              object
anken_id                        object
kohosha_id                      object
linked_shokikosho_id            object
tsr_code                        object
kigyo_name                      object
annual_income                   object
layer                           object
initial_contact_date    datetime64[ns]
shoki_setteibi          datetime64[ns]
shoki_jisshibi          datetime64[ns]
hon_setteibi            datetime64[ns]
hon_yoteibi             datetime64[ns]
hon_jisshibi            datetime64[ns]
kohosha_tanto                   object
kigyo_tanto                     object
hon_seq                         object
shoki_seq                       object
jisshibi_sa                     object
yomi                            object
yomi_final                      object
uriage                          object
yakushoku_class                 object
first_mendan_tanto              object
hanjokin                        object
APsource                 

In [ ]:
# ==========================================
# セル2: Q_masterのマージ（メソッドチェーンでスッキリと）
# ==========================================

# 設定日のマージとリネームを1つの処理で繋げて行います
honkosho_data = (
    honkosho_data.merge(q_master_subset, left_on='hon_setteibi', right_on='日付', how='left')
    .drop(columns=['日付'])
    .rename(columns={'Q': 'setteibi_Q', '月': 'setteibi_月', '営業日': 'setteibi_営業日', '同営業日比較': 'setteibi_同営業日比較'})
)

# 実施日のマージとリネームも同様に行います
honkosho_data = (
    honkosho_data.merge(q_master_subset, left_on='hon_jisshibi', right_on='日付', how='left')
    .drop(columns=['日付'])
    .rename(columns={'Q': 'jisshibi_Q', '月': 'jisshibi_月', '営業日': 'jisshibi_営業日', '同営業日比較': 'jisshibi_同営業日比較'})
)

In [ ]:
# ==========================================
# セル3: master2（社員マスタ）のマージ
# ==========================================

# 候補者担当者情報のマージ
honkosho_data = (
    honkosho_data.merge(master2_subset, left_on=['setteibi_Q', 'kohosha_tanto'], right_on=['Q', 'sei_plus'], how='left')
    .drop(columns=['Q', 'sei_plus'])
    .rename(columns={'職種': '候担_職種', 'ロンザン所属フラグ': '候担_ロンザン所属フラグ', 'レイヤー': '候担_レイヤー'})
)

# 企業担当者情報のマージ
honkosho_data = (
    honkosho_data.merge(master2_subset, left_on=['setteibi_Q', 'kigyo_tanto'], right_on=['Q', 'sei_plus'], how='left')
    .drop(columns=['Q', 'sei_plus'])
    .rename(columns={'職種': '企担_職種', 'ロンザン所属フラグ': '企担_ロンザン所属フラグ', 'レイヤー': '企担_レイヤー'})
)



In [ ]:
# ==========================================
# 新しいカラム「組み手」を追加する処理
# ==========================================

# 1. 組み手を判定するための独自のルール（関数）を作ります
def judge_kumite(row):
    # 安全に判定するため、値を一度「文字」として取り出します
    koho_flag = str(row['候担_ロンザン所属フラグ'])
    kigyo_flag = str(row['企担_ロンザン所属フラグ'])
    
    # ルール1: 両方とも '1' の場合（"1.0" となっているケースも考慮して "1" が含まれるかチェックします）
    if '1' in koho_flag and '1' in kigyo_flag:
        return '両手'
    
    # ルール2: どちらか片方でも '1' の場合（上の条件を抜けたものがここに来ます）
    elif '1' in koho_flag or '1' in kigyo_flag:
        return '片手'
    
    # ルール3: それ以外（両方 '0' や、空欄など）の場合
    else:
        return '無効'

# 2. 作ったルール（judge_kumite）を、データフレームの行ごと（axis=1）に適用（apply）します
honkosho_data['組み手'] = honkosho_data.apply(judge_kumite, axis=1)


In [ ]:
display(honkosho_data.head())

,id,anken_id,kohosha_id,linked_shokikosho_id,tsr_code,kigyo_name,annual_income,layer,initial_contact_date,shoki_setteibi,shoki_jisshibi,hon_setteibi,hon_yoteibi,hon_jisshibi,kohosha_tanto,kigyo_tanto,hon_seq,shoki_seq,jisshibi_sa,yomi,yomi_final,uriage,yakushoku_class,first_mendan_tanto,hanjokin,APsource,sai_flg2,setteibi_Q,setteibi_月,setteibi_営業日,setteibi_同営業日比較,jisshibi_Q,jisshibi_月,jisshibi_営業日,jisshibi_同営業日比較,候担_職種,候担_ロンザン所属フラグ,候担_レイヤー,企担_職種,企担_ロンザン所属フラグ,企担_レイヤー,組み手
0,1,1,663,-1,410147966,（株）ミールケア,NaN,None,NaT,NaT,NaT,2016-07-05,2016-07-13,2016-07-13,角田隆,北健,1,<NA>,<NA>,オチ（双）,成約,84.0,None,None,,パートナー紹介,2,19-4Q,2016-07-01,3,同営業日,19-4Q,2016-07-01,9,同営業日,ミドル企業,1,None,NaN,NaN,NaN,片手
1,2,2,165,-1,292368810,（株）大治,NaN,None,NaT,NaT,NaT,2016-07-05,2016-07-15,2016-07-15,角田隆,嵯峨優,1,<NA>,<NA>,D-,D-,59.0,None,None,,パートナー紹介,2,19-4Q,2016-07-01,3,同営業日,19-4Q,2016-07-01,11,同営業日,ミドル企業,1,None,NaN,NaN,NaN,片手
2,3,3,664,-1,422112860,ネクストエナジー・アンド・リソース（株）,NaN,None,NaT,NaT,NaT,2016-07-05,2016-07-12,2016-07-12,大矢裕,岡大,1,<NA>,<NA>,オチ（企）,オチ（企）,116.0,None,None,,顧問名鑑登録者,2,19-4Q,2016-07-01,3,同営業日,19-4Q,2016-07-01,8,同営業日,ミドル企業,1,None,NaN,NaN,NaN,片手
3,4,4,455,1284,400652951,（株）メニコン,7200.0,None,2016-02-24,2016-02-24,2016-02-24,2016-07-05,2016-09-20,2016-09-20,大仲研,大仲研,1,1,132,D,成約,710.0,None,大仲研,,その他,2,19-4Q,2016-07-01,3,同営業日,19-4Q,2016-09-01,53,,ミドル企業,1,None,ミドル企業,1,None,両手
4,5,5,665,-1,422112860,ネクストエナジー・アンド・リソース（株）,NaN,None,NaT,NaT,NaT,2016-07-05,2016-07-12,2016-07-12,角田隆,岡大,1,<NA>,<NA>,オチ（企）,オチ（企）,116.0,None,None,,パートナー紹介,2,19-4Q,2016-07-01,3,同営業日,19-4Q,2016-07-01,8,同営業日,ミドル企業,1,None,NaN,NaN,NaN,片手


In [ ]:
# ==========================================
# セル4: データの抽出、整形、縦積み（関数化で劇的にコード削減！）
# ==========================================

# 1. 対象データを一度だけ抽出します
base_hon = honkosho_data.copy()

# 2. 共通で引き継ぐカラム名と、新しく付与するキレイなカラム名のリスト
common_cols = [
    'id', 'kohosha_id', 'tsr_code', 'anken_id', 'APsource', 'kohosha_tanto', 'kigyo_tanto', # ← 'tsr_code' を追加！
    '候担_職種', '候担_ロンザン所属フラグ', '候担_レイヤー', 
    '企担_職種', '企担_ロンザン所属フラグ', '企担_レイヤー', '組み手'
]
clean_cols = [
    'Q', '月', '営業日', '同営業日', '日付',           
    'KPI_ID', '候補者ID', 'tsr_code', '案件ID', 'APソース', '候補者担当', '企業担当', # ← ここにも 'tsr_code' を追加！
    '候担_職種', '候担_ロンザン所属フラグ', '候担_レイヤー',       
    '企担_職種', '企担_ロンザン所属フラグ', '企担_レイヤー', '組み手'   
]

# ==========================================
# ★ここが魔法の工場（関数）です！
# 条件(condition) と type名(type_name) を渡すと、自動で整形されたデータを作ります
# ==========================================
def make_df(condition, type_name, date_type='settei'):
    # 設定日ベースか、実施日ベースかで使う日付カラムを自動で切り替えます
    if date_type == 'settei':
        date_cols = ['setteibi_Q', 'setteibi_月', 'setteibi_営業日', 'setteibi_同営業日比較', 'hon_setteibi']
    else:
        date_cols = ['jisshibi_Q', 'jisshibi_月', 'jisshibi_営業日', 'jisshibi_同営業日比較', 'hon_jisshibi']
        
    # 条件で絞り込み、必要なカラムだけ抽出
    df_temp = base_hon[condition][date_cols + common_cols].copy()
    
    # リネームしてtypeをセット
    df_temp.columns = clean_cols
    df_temp['type'] = type_name
    
    return df_temp

# ==========================================
# 3. 必要なtypeのデータフレームをどんどん作って、リスト(dfs)に放り込みます
# ==========================================
dfs = []
mask_all = pd.Series(True, index=base_hon.index) # 「全件」を表すおまじない

# '半常勤' と一致するもの
mask_han = base_hon['hanjokin'] == '半常勤'
# それ以外（~ は「否定（Not）」を意味します。空白やNaNもこちらに含まれます）
mask_not_han = ~mask_han

# --- 基本 ---
dfs.append(make_df(mask_all & mask_not_han, 'hon_settei'))
dfs.append(make_df(mask_all & mask_not_han, 'hon_jisshi', date_type='jisshi')) # 実施データも残す

# --- 新規/再 ---
dfs.append(make_df((base_hon['sai_flg2'] != 2) & mask_not_han, 'hon_settei_shin'))
dfs.append(make_df((base_hon['sai_flg2'] == 2) & mask_not_han, 'hon_settei_sai'))

# --- 片手 ---
dfs.append(make_df((base_hon['組み手'] == '片手') & mask_not_han, 'hon_settei_kata'))
dfs.append(make_df((base_hon['組み手'] == '片手') & (base_hon['sai_flg2'] != 2) & mask_not_han, 'hon_settei_kata_shin'))
dfs.append(make_df((base_hon['組み手'] == '片手') & (base_hon['sai_flg2'] == 2) & mask_not_han, 'hon_settei_kata_sai'))

# --- 両手 ---
dfs.append(make_df((base_hon['組み手'] == '両手') & mask_not_han, 'hon_settei_ryo'))
dfs.append(make_df((base_hon['組み手'] == '両手') & (base_hon['sai_flg2'] != 2) & mask_not_han, 'hon_settei_ryo_shin'))
dfs.append(make_df((base_hon['組み手'] == '両手') & (base_hon['sai_flg2'] == 2) & mask_not_han, 'hon_settei_ryo_sai'))

# --- 追加分（社数・人数） ---
dfs.append(make_df(mask_all & mask_not_han, 'hon_settei_sha'))
dfs.append(make_df(mask_all & mask_not_han, 'hon_settei_nin'))

# --- 企業担当視点 ---
mask_kigyo = mask_all & (base_hon['企担_ロンザン所属フラグ'].astype(str).str.contains('1'))

dfs.append(make_df(mask_kigyo & mask_not_han, 'kigyo_settei'))
dfs.append(make_df(mask_kigyo & mask_not_han, 'kigyo_settei_sha'))
dfs.append(make_df(mask_kigyo & mask_not_han, 'kigyo_settei_nin'))

# ==========================================
# 3.1 新規追加：半常勤の項目の集計
# ==========================================
# --- 基本（半常勤） ---
dfs.append(make_df(mask_all & mask_han, 'hon_settei_han'))
# ※ もし hon_jisshi_han も必要であれば同様に追加可能です

# --- 新規/再（半常勤） ---
dfs.append(make_df((base_hon['sai_flg2'] != 2) & mask_han, 'hon_settei_han_shin'))
dfs.append(make_df((base_hon['sai_flg2'] == 2) & mask_han, 'hon_settei_han_sai'))

# --- 片手（半常勤） ---
dfs.append(make_df((base_hon['組み手'] == '片手') & mask_han, 'hon_settei_han_kata'))
dfs.append(make_df((base_hon['組み手'] == '片手') & (base_hon['sai_flg2'] != 2) & mask_han, 'hon_settei_han_kata_shin'))
dfs.append(make_df((base_hon['組み手'] == '片手') & (base_hon['sai_flg2'] == 2) & mask_han, 'hon_settei_han_kata_sai'))

# --- 両手（半常勤） ---
dfs.append(make_df((base_hon['組み手'] == '両手') & mask_han, 'hon_settei_han_ryo'))
dfs.append(make_df((base_hon['組み手'] == '両手') & (base_hon['sai_flg2'] != 2) & mask_han, 'hon_settei_han_ryo_shin'))
dfs.append(make_df((base_hon['組み手'] == '両手') & (base_hon['sai_flg2'] == 2) & mask_han, 'hon_settei_han_ryo_sai'))

# --- 追加分（社数・人数）（半常勤） ---
dfs.append(make_df(mask_all & mask_han, 'hon_settei_han_sha'))
dfs.append(make_df(mask_all & mask_han, 'hon_settei_han_nin'))

# --- 企業担当視点（半常勤） ---
dfs.append(make_df(mask_kigyo & mask_han, 'kigyo_settei_han'))
dfs.append(make_df(mask_kigyo & mask_han, 'kigyo_settei_han_sha'))
dfs.append(make_df(mask_kigyo & mask_han, 'kigyo_settei_han_nin'))

# 今後「hon_settei_kigyo」などが増える場合は、ここに dfs.append(...) を1行足すだけでOKです！

# ==========================================
# 4. まとめて縦にガッチャンコ
# ==========================================
hon_combined_data = pd.concat(dfs, ignore_index=True)
hon_combined_data['value'] = 1

# ==========================================
# ★集計マジック：指標によって「何を数えるか」を自動で切り替える仕込み
# ==========================================
# ① 全行に、絶対に重複しない「連番」を振る（通常の count と同じ挙動にさせるため）
hon_combined_data['calc_target'] = range(len(hon_combined_data))

# ② 「_sha」で終わる type は、数えるターゲットを「tsr_code」に上書き
mask_sha = hon_combined_data['type'].str.endswith('_sha', na=False)
hon_combined_data.loc[mask_sha, 'calc_target'] = hon_combined_data.loc[mask_sha, 'tsr_code']

# ③ 「_nin」で終わる type は、数えるターゲットを「候補者ID」に上書き
mask_nin = hon_combined_data['type'].str.endswith('_nin', na=False)
hon_combined_data.loc[mask_nin, 'calc_target'] = hon_combined_data.loc[mask_nin, '候補者ID']  # ← ここを日本語の '候補者ID' にしました！


# 結合後のデータを確認
print("結合後のデータの行数と列数:", hon_combined_data.shape)
display(hon_combined_data.head())

結合後のデータの行数と列数: (228813, 22)


C:\Users\suehara\AppData\Local\Temp\ipykernel_22204\2302710035.py:127: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['410147966' '292368810' '422112860' ... '313710481' '430142790'
 '016807820']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  hon_combined_data.loc[mask_sha, 'calc_target'] = hon_combined_data.loc[mask_sha, 'tsr_code']


,Q,月,営業日,同営業日,日付,KPI_ID,候補者ID,tsr_code,案件ID,APソース,候補者担当,企業担当,候担_職種,候担_ロンザン所属フラグ,候担_レイヤー,企担_職種,企担_ロンザン所属フラグ,企担_レイヤー,組み手,type,value,calc_target
0,19-4Q,2016-07-01,3,同営業日,2016-07-05,1,663,410147966,1,パートナー紹介,角田隆,北健,ミドル企業,1,None,NaN,NaN,NaN,片手,hon_settei,1,0
1,19-4Q,2016-07-01,3,同営業日,2016-07-05,2,165,292368810,2,パートナー紹介,角田隆,嵯峨優,ミドル企業,1,None,NaN,NaN,NaN,片手,hon_settei,1,1
2,19-4Q,2016-07-01,3,同営業日,2016-07-05,3,664,422112860,3,顧問名鑑登録者,大矢裕,岡大,ミドル企業,1,None,NaN,NaN,NaN,片手,hon_settei,1,2
3,19-4Q,2016-07-01,3,同営業日,2016-07-05,4,455,400652951,4,その他,大仲研,大仲研,ミドル企業,1,None,ミドル企業,1,None,両手,hon_settei,1,3
4,19-4Q,2016-07-01,3,同営業日,2016-07-05,5,665,422112860,5,パートナー紹介,角田隆,岡大,ミドル企業,1,None,NaN,NaN,NaN,片手,hon_settei,1,4


In [ ]:
mask_sha

0         False
1         False
2         False
3         False
4         False
          ...  
228808    False
228809    False
228810    False
228811    False
228812    False
Name: type, Length: 228813, dtype: bool

In [ ]:

hon_combined_data.loc[mask_sha, 'calc_target'] = hon_combined_data.loc[mask_sha, 'tsr_code']

# ③ 「_nin」で終わる type は、数えるターゲットを「kohosha_id」に上書き
mask_nin = hon_combined_data['type'].str.endswith('_nin', na=False)
hon_combined_data.loc[mask_nin, 'calc_target'] = hon_combined_data.loc[mask_nin, '候補者ID']


# 結合後のデータを確認
print("結合後のデータの行数と列数:", hon_combined_data.shape)
display(hon_combined_data.head())

結合後のデータの行数と列数: (228813, 22)


,Q,月,営業日,同営業日,日付,KPI_ID,候補者ID,tsr_code,案件ID,APソース,候補者担当,企業担当,候担_職種,候担_ロンザン所属フラグ,候担_レイヤー,企担_職種,企担_ロンザン所属フラグ,企担_レイヤー,組み手,type,value,calc_target
0,19-4Q,2016-07-01,3,同営業日,2016-07-05,1,663,410147966,1,パートナー紹介,角田隆,北健,ミドル企業,1,None,NaN,NaN,NaN,片手,hon_settei,1,0
1,19-4Q,2016-07-01,3,同営業日,2016-07-05,2,165,292368810,2,パートナー紹介,角田隆,嵯峨優,ミドル企業,1,None,NaN,NaN,NaN,片手,hon_settei,1,1
2,19-4Q,2016-07-01,3,同営業日,2016-07-05,3,664,422112860,3,顧問名鑑登録者,大矢裕,岡大,ミドル企業,1,None,NaN,NaN,NaN,片手,hon_settei,1,2
3,19-4Q,2016-07-01,3,同営業日,2016-07-05,4,455,400652951,4,その他,大仲研,大仲研,ミドル企業,1,None,ミドル企業,1,None,両手,hon_settei,1,3
4,19-4Q,2016-07-01,3,同営業日,2016-07-05,5,665,422112860,5,パートナー紹介,角田隆,岡大,ミドル企業,1,None,NaN,NaN,NaN,片手,hon_settei,1,4


In [ ]:
# ==========================================
# 1. あなたが並べたい理想の順番を「リスト」で定義します
# 今後KPIが増えたら、ここにどんどんカンマ区切りで追記していくだけでOKです！
# ==========================================
type_order = [
    'hon_settei',
    'hon_settei_shin',
    'hon_settei_sai',
    'hon_settei_kata',
    'hon_settei_kata_shin',
    'hon_settei_kata_sai',
    'hon_settei_ryo',
    'hon_settei_ryo_shin',
    'hon_settei_ryo_sai',
    'hon_settei_sha',
    'hon_settei_nin',
    'kigyo_settei',
    'kigyo_settei_sha',
    'kigyo_settei_nin',

    'hon_settei_han',
    'hon_settei_han_shin',
    'hon_settei_han_sai',
    'hon_settei_han_kata',
    'hon_settei_han_kata_shin',
    'hon_settei_han_kata_sai',
    'hon_settei_han_ryo',
    'hon_settei_han_ryo_shin',
    'hon_settei_han_ryo_sai',
    'hon_settei_han_sha',
    'hon_settei_han_nin',
    'kigyo_settei_han',
    'kigyo_settei_han_sha',
    'kigyo_settei_han_nin'
]

# ==========================================
# 2. hon_combined_data の 'type' カラムに、上で作った「独自の順番（ルール）」を記憶させます
# ==========================================
hon_combined_data['type'] = pd.Categorical(
    hon_combined_data['type'], 
    categories=type_order, 
    ordered=True  # 「この順番に意味があるよ（順序付きだよ）」と教えてあげます
)

In [ ]:
# ==========================================
# セル5: ピボットテーブルの作成（全フェーズ一括処理）
# ==========================================
import pandas as pd

# 1. 高速化・メモリ最適化された集計エンジンの定義
def aggregate_phase_data(df: pd.DataFrame, valid_members: list, type_order: list) -> pd.DataFrame:
    """
    指定されたフェーズのデータフレームに対し、'Q', '月', 'Q_同営業日' の3軸で
    ピボット集計を行い、横に結合したデータフレームを高速に生成する。
    """
    # (前回提示した関数の内部コードをここに配置)
    df = df.copy(deep=False) 
    df['Q_同営業日'] = df['Q'].astype(str) + "同営業日"
    df['全体'] = '合計' # ★ここを追加
    
    time_columns = ['Q', '月', 'Q_同営業日']
    agg_configs = [
        ("全体", "全体", False), # ★ここを追加
        ("APソース", "APソース", False),
        ("候担_職種", "職種", False),
        ("候担_レイヤー", "レイヤー", False),
        ("候補者担当", "担当", True),
        ("企担_職種", "企担_職種", False),
        ("企業担当", "企業担当", True)
    ]
    
    time_axis_dfs = []
    
    for time_col in time_columns:
        axis_dfs = []
        for col, axis_name, need_filter in agg_configs:
            pivot = df.pivot_table(
                index=["type", col], columns=time_col,
                aggfunc="nunique", values="calc_target"
            ).fillna(0).reset_index()
            
            if need_filter:
                pivot = pivot[pivot[col].isin(valid_members)]
                
            pivot = pivot.rename(columns={col: '項目名'})
            pivot.insert(1, '集計軸', axis_name)
            axis_dfs.append(pivot)
            
        combined_axis = pd.concat(axis_dfs, ignore_index=True)
        combined_axis = combined_axis.set_index(["type", "集計軸", "項目名"])
        time_axis_dfs.append(combined_axis)

    final_df = pd.concat(time_axis_dfs, axis=1).fillna(0).reset_index()
    final_df['type'] = pd.Categorical(final_df['type'], categories=type_order, ordered=True)
    
    numeric_cols = final_df.columns.difference(['type', '集計軸', '項目名'])
    final_df = final_df[final_df[numeric_cols].sum(axis=1) > 0]
    
    final_df = final_df.sort_values(by=['type', '集計軸', '項目名'])
    final_df.columns.name = None
    
    return final_df

# ==========================================
# 2. 実行および各フェーズのガッチャンコ
# ==========================================
# 担当者マスタのリスト化 (ベクタライズされたフィルタリング用)
valid_members = master2['sei_plus'].dropna().unique()

# 関数を呼び出して一気に処理 (ローカルスコープで処理されるためメモリ効率が高い)
hon_pivot_df   = aggregate_phase_data(hon_combined_data, valid_members, type_order)

# 確認表示
print("【初期交渉データ】")
display(shoki_pivot_df.head(3))
print("【本交渉データ】")
display(hon_pivot_df.head(3))
print("【成約データ】")


C:\Users\suehara\AppData\Local\Temp\ipykernel_22204\3826215038.py:33: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot = df.pivot_table(
C:\Users\suehara\AppData\Local\Temp\ipykernel_22204\3826215038.py:33: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot = df.pivot_table(
C:\Users\suehara\AppData\Local\Temp\ipykernel_22204\3826215038.py:33: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot = df.pivot_table(
C:\Users\suehara\AppData\Local\Temp\ipykernel_22204\3826215038.py:33: FutureWarning: The

【初期交渉データ】


,type,集計軸,項目名,18-1Q,18-2Q,18-3Q,18-4Q,19-1Q,19-2Q,19-3Q,19-4Q,20-1Q,20-2Q,20-3Q,20-4Q,21-1Q,21-2Q,21-3Q,21-4Q,22-1Q,22-2Q,22-3Q,22-4Q,23-1Q,23-2Q,23-3Q,23-4Q,24-1Q,24-2Q,24-3Q,24-4Q,25-1Q,25-2Q,25-3Q,25-4Q,26-1Q,26-2Q,26-3Q,26-4Q,27-1Q,27-2Q,27-3Q,27-4Q,28-1Q,28-2Q,28-3Q,28-4Q,29-1Q,29-2Q,29-3Q,2014-11-01 00:00:00,2014-12-01 00:00:00,2015-01-01 00:00:00,2015-02-01 00:00:00,2015-03-01 00:00:00,2015-04-01 00:00:00,2015-07-01 00:00:00,2015-10-01 00:00:00,2015-11-01 00:00:00,2015-12-01 00:00:00,2016-01-01 00:00:00,2016-02-01 00:00:00,2016-03-01 00:00:00,2016-04-01 00:00:00,2016-05-01 00:00:00,2016-06-01 00:00:00,2016-07-01 00:00:00,2016-08-01 00:00:00,2016-09-01 00:00:00,2016-10-01 00:00:00,2016-11-01 00:00:00,2016-12-01 00:00:00,2017-01-01 00:00:00,2017-02-01 00:00:00,2017-03-01 00:00:00,2017-04-01 00:00:00,2017-05-01 00:00:00,2017-06-01 00:00:00,2017-07-01 00:00:00,2017-08-01 00:00:00,2017-09-01 00:00:00,2017-10-01 00:00:00,2017-11-01 00:00:00,2017-12-01 00:00:00,2018-01-01 00:00:00,2018-02-01 00:00:00,2018-03-01 00:00:00,2018-04-01 00:00:00,2018-05-01 00:00:00,2018-06-01 00:00:00,2018-07-01 00:00:00,2018-08-01 00:00:00,2018-09-01 00:00:00,2018-10-01 00:00:00,2018-11-01 00:00:00,2018-12-01 00:00:00,2019-01-01 00:00:00,2019-02-01 00:00:00,2019-03-01 00:00:00,2019-04-01 00:00:00,...,2022-03-01 00:00:00,2022-04-01 00:00:00,2022-05-01 00:00:00,2022-06-01 00:00:00,2022-07-01 00:00:00,2022-08-01 00:00:00,2022-09-01 00:00:00,2022-10-01 00:00:00,2022-11-01 00:00:00,2022-12-01 00:00:00,2023-01-01 00:00:00,2023-02-01 00:00:00,2023-03-01 00:00:00,2023-04-01 00:00:00,2023-05-01 00:00:00,2023-06-01 00:00:00,2023-07-01 00:00:00,2023-08-01 00:00:00,2023-09-01 00:00:00,2023-10-01 00:00:00,2023-11-01 00:00:00,2023-12-01 00:00:00,2024-01-01 00:00:00,2024-02-01 00:00:00,2024-03-01 00:00:00,2024-04-01 00:00:00,2024-05-01 00:00:00,2024-06-01 00:00:00,2024-07-01 00:00:00,2024-08-01 00:00:00,2024-09-01 00:00:00,2024-10-01 00:00:00,2024-11-01 00:00:00,2024-12-01 00:00:00,2025-01-01 00:00:00,2025-02-01 00:00:00,2025-03-01 00:00:00,2025-04-01 00:00:00,2025-05-01 00:00:00,2025-06-01 00:00:00,2025-07-01 00:00:00,2025-08-01 00:00:00,2025-09-01 00:00:00,2025-10-01 00:00:00,2025-11-01 00:00:00,2025-12-01 00:00:00,2026-01-01 00:00:00,2026-02-01 00:00:00,2026-03-01 00:00:00,2026-04-01 00:00:00,2026-05-01 00:00:00,2026-06-01 00:00:00,18-1Q同営業日,18-2Q同営業日,18-3Q同営業日,18-4Q同営業日,19-1Q同営業日,19-2Q同営業日,19-3Q同営業日,19-4Q同営業日,20-1Q同営業日,20-2Q同営業日,20-3Q同営業日,20-4Q同営業日,21-1Q同営業日,21-2Q同営業日,21-3Q同営業日,21-4Q同営業日,22-1Q同営業日,22-2Q同営業日,22-3Q同営業日,22-4Q同営業日,23-1Q同営業日,23-2Q同営業日,23-3Q同営業日,23-4Q同営業日,24-1Q同営業日,24-2Q同営業日,24-3Q同営業日,24-4Q同営業日,25-1Q同営業日,25-2Q同営業日,25-3Q同営業日,25-4Q同営業日,26-1Q同営業日,26-2Q同営業日,26-3Q同営業日,26-4Q同営業日,27-1Q同営業日,27-2Q同営業日,27-3Q同営業日,27-4Q同営業日,28-1Q同営業日,28-2Q同営業日,28-3Q同営業日,28-4Q同営業日,29-1Q同営業日,29-2Q同営業日,29-3Q同営業日,nan同営業日
0,shoki_settei,APソース,SMAP,0,2,0,0,0,10,43,17,36,58,40,41,36,50,42,76,71,97,112,101,106,96,180,144,231,265,185,334,183,183,203,508,876,1173,928,930,1128,1192,1282,1414,1429,1426,1545,1568,1622,1577,994,0,0,1,1,0,0,0,0,0,0,6,2,2,19,9,16,9,7,1,8,20,8,17,18,22,16,14,10,15,14,12,12,18,6,16,14,20,3,7,32,30,25,20,22,32,18,27,29,43,32,...,74,64,69,70,115,160,230,273,320,297,258,458,467,250,357,311,276,337,290,366,398,402,398,401,398,383,454,439,508,350,532,490,494,470,490,438,516,508,529,467,628,428,510,504,540,597,467,494,597,487,414,93,0,2,0,0,0,10,43,17,36,58,40,41,36,50,42,76,71,97,112,101,106,96,180,144,231,265,185,334,183,183,203,508,876,1173,928,930,1128,1192,1282,1414,1429,1426,1545,1568,1622,1577,994,1
1,shoki_settei,APソース,その他,0,0,0,1,1,11,26,30,57,118,68,142,77,132,115,106,76,64,61,72,65,72,83,72,96,114,87,59,64,64,48,66,54,27,45,44,62,85,82,93,79,95,105,132,84,91,51,0,0,0,0,0,0,1,0,0,1,3,2,6,5,11,10,21,6,3,22,24,11,38,41,39,27,19,23,35,28,78,29,27,21,46,49,37,37,47,31,48,32,24,32,20,27,21,24,18,20,...,20,13,21,13,29,27,10,19,20,18,20,2,2,13,9,23,15,15,13,20,27,16,27,26,32,32,31,20,23,26,43,28,29,23,37,27,33,36,36,32,38,41,51,29,25,31,31,36,23,23,25,3,0,0,0,1,1,

【本交渉データ】


,type,集計軸,項目名,19-1Q,19-4Q,20-1Q,20-2Q,20-3Q,20-4Q,21-1Q,21-2Q,21-3Q,21-4Q,22-1Q,22-2Q,22-3Q,22-4Q,23-1Q,23-2Q,23-3Q,23-4Q,24-1Q,24-2Q,24-3Q,24-4Q,25-1Q,25-2Q,25-3Q,25-4Q,26-1Q,26-2Q,26-3Q,26-4Q,27-1Q,27-2Q,27-3Q,27-4Q,28-1Q,28-2Q,28-3Q,28-4Q,29-1Q,29-2Q,29-3Q,2015-10-01 00:00:00,2016-07-01 00:00:00,2016-08-01 00:00:00,2016-09-01 00:00:00,2016-10-01 00:00:00,2016-11-01 00:00:00,2016-12-01 00:00:00,2017-01-01 00:00:00,2017-02-01 00:00:00,2017-03-01 00:00:00,2017-04-01 00:00:00,2017-05-01 00:00:00,2017-06-01 00:00:00,2017-07-01 00:00:00,2017-08-01 00:00:00,2017-09-01 00:00:00,2017-10-01 00:00:00,2017-11-01 00:00:00,2017-12-01 00:00:00,2018-01-01 00:00:00,2018-02-01 00:00:00,2018-03-01 00:00:00,2018-04-01 00:00:00,2018-05-01 00:00:00,2018-06-01 00:00:00,2018-07-01 00:00:00,2018-08-01 00:00:00,2018-09-01 00:00:00,2018-10-01 00:00:00,2018-11-01 00:00:00,2018-12-01 00:00:00,2019-01-01 00:00:00,2019-02-01 00:00:00,2019-03-01 00:00:00,2019-04-01 00:00:00,2019-05-01 00:00:00,2019-06-01 00:00:00,2019-07-01 00:00:00,2019-08-01 00:00:00,2019-09-01 00:00:00,2019-10-01 00:00:00,2019-11-01 00:00:00,2019-12-01 00:00:00,2020-01-01 00:00:00,2020-02-01 00:00:00,2020-03-01 00:00:00,2020-04-01 00:00:00,2020-05-01 00:00:00,2020-06-01 00:00:00,2020-07-01 00:00:00,2020-08-01 00:00:00,2020-09-01 00:00:00,2020-10-01 00:00:00,2020-11-01 00:00:00,2020-12-01 00:00:00,2021-01-01 00:00:00,...,2021-09-01 00:00:00,2021-10-01 00:00:00,2021-11-01 00:00:00,2021-12-01 00:00:00,2022-01-01 00:00:00,2022-02-01 00:00:00,2022-03-01 00:00:00,2022-04-01 00:00:00,2022-05-01 00:00:00,2022-06-01 00:00:00,2022-07-01 00:00:00,2022-08-01 00:00:00,2022-09-01 00:00:00,2022-10-01 00:00:00,2022-11-01 00:00:00,2022-12-01 00:00:00,2023-01-01 00:00:00,2023-02-01 00:00:00,2023-03-01 00:00:00,2023-04-01 00:00:00,2023-05-01 00:00:00,2023-06-01 00:00:00,2023-07-01 00:00:00,2023-08-01 00:00:00,2023-09-01 00:00:00,2023-10-01 00:00:00,2023-11-01 00:00:00,2023-12-01 00:00:00,2024-01-01 00:00:00,2024-02-01 00:00:00,2024-03-01 00:00:00,2024-04-01 00:00:00,2024-05-01 00:00:00,2024-06-01 00:00:00,2024-07-01 00:00:00,2024-08-01 00:00:00,2024-09-01 00:00:00,2024-10-01 00:00:00,2024-11-01 00:00:00,2024-12-01 00:00:00,2025-01-01 00:00:00,2025-02-01 00:00:00,2025-03-01 00:00:00,2025-04-01 00:00:00,2025-05-01 00:00:00,2025-06-01 00:00:00,2025-07-01 00:00:00,2025-08-01 00:00:00,2025-09-01 00:00:00,2025-10-01 00:00:00,2025-11-01 00:00:00,2025-12-01 00:00:00,2026-01-01 00:00:00,2026-02-01 00:00:00,2026-03-01 00:00:00,2026-04-01 00:00:00,2026-05-01 00:00:00,2026-06-01 00:00:00,19-1Q同営業日,19-4Q同営業日,20-1Q同営業日,20-2Q同営業日,20-3Q同営業日,20-4Q同営業日,21-1Q同営業日,21-2Q同営業日,21-3Q同営業日,21-4Q同営業日,22-1Q同営業日,22-2Q同営業日,22-3Q同営業日,22-4Q同営業日,23-1Q同営業日,23-2Q同営業日,23-3Q同営業日,23-4Q同営業日,24-1Q同営業日,24-2Q同営業日,24-3Q同営業日,24-4Q同営業日,25-1Q同営業日,25-2Q同営業日,25-3Q同営業日,25-4Q同営業日,26-1Q同営業日,26-2Q同営業日,26-3Q同営業日,26-4Q同営業日,27-1Q同営業日,27-2Q同営業日,27-3Q同営業日,27-4Q同営業日,28-1Q同営業日,28-2Q同営業日,28-3Q同営業日,28-4Q同営業日,29-1Q同営業日,29-2Q同営業日,29-3Q同営業日,nan同営業日
28,hon_settei,APソース,SMAP,0,63,57,34,44,34,37,27,21,34,52,73,94,84,99,76,68,97,93,135,115,133,93,88,105,124,234,250,347,335,373,343,449,463,424,478,483,533,521,543,400,0,16,26,21,15,25,18,10,16,9,17,8,17,21,6,7,10,14,13,7,11,9,6,7,8,12,10,10,21,21,14,13,32,28,31,30,33,31,38,13,34,39,28,37,21,17,22,27,18,40,31,26,32,36,25,39,...,38,42,38,18,25,34,29,33,45,29,36,41,45,72,83,81,107,75,69,117,109,123,115,134,79,117,165,101,135,117,88,151,184,118,176,156,122,161,124,146,173,169,136,178,167,137,239,168,120,172,185,177,193,174,163,189,176,35,0,63,57,34,44,34,37,27,21,34,52,73,94,84,99,76,68,97,93,135,115,133,93,88,105,124,234,250,347,335,373,343,449,463,424,478,483,533,521,543,400,0
29,hon_settei,APソース,その他,0,13,45,36,48,29,38,38,23,31,30,19,33,41,46,51,53,64,64,76,74,51,52,51,33,43,29,28,33,38,33,28,43,28,35,48,46,59,40,39,38,0,7,3,3,12,14,19,13,12,13,18,17,12,11,8,9,16,12,10,14,12,13,11,5,6,15,8,8,12,15,3,4,8,7,17,7,9,17,13,11,10,23,13,19,23,10,19,20,14,28,17,18,22,25,17,21,...,21,16,25,11,20,13,19,11,12,9,14,20,9,11,8,11,22,4,1,

【成約データ】


In [ ]:
# ==========================================
# 最終工程: データの統合とスプレッドシート書き出し
# ==========================================

shoki_pivot_df['type'] = shoki_pivot_df['type'].astype(str)
hon_pivot_df['type'] = hon_pivot_df['type'].astype(str)

# 1. 初期交渉と本交渉のデータを縦にガッチャンコ！
final_all_df = pd.concat([shoki_pivot_df, hon_pivot_df], ignore_index=True)


# 2. 全KPI（type）の並び順リストを定義
full_type_order = [
    'shoki_settei', 'shoki_jisshi',
    'hon_settei', 'hon_jisshi',
    'hon_settei_shin', 'hon_settei_sai',
    'hon_settei_kata', 'hon_settei_kata_shin', 'hon_settei_kata_sai',
    'hon_settei_ryo', 'hon_settei_ryo_shin', 'hon_settei_ryo_sai',
    'hon_settei_sha','hon_settei_nin','kigyo_settei','kigyo_settei_sha','kigyo_settei_nin',

    'hon_settei_han',
    'hon_settei_han_shin','hon_settei_han_sai',
    'hon_settei_han_kata','hon_settei_han_kata_shin','hon_settei_han_kata_sai',
    'hon_settei_han_ryo','hon_settei_han_ryo_shin','hon_settei_han_ryo_sai',
    'hon_settei_han_sha','hon_settei_han_nin',
    'kigyo_settei_han','kigyo_settei_han_sha','kigyo_settei_han_nin'
]

# 3. 改めて並び替え（ソート）を適用
final_all_df['type'] = pd.Categorical(final_all_df['type'], categories=full_type_order, ordered=True)


# 数値列（Qの列）の空欄を 0 で埋める
numeric_cols = final_all_df.columns.difference(['type', '集計軸', '項目名'])
final_all_df[numeric_cols] = final_all_df[numeric_cols].fillna(0)

# 並び替え実行
final_all_df = final_all_df.sort_values(by=['type', '集計軸', '項目名'])
final_all_df.columns.name = None




In [ ]:
# # --- 修正・改善した書き出し処理（究極の完全版） ---

# # 1. ★超重要★ 「表の中身」を先にすべて文字列に変換
# # カテゴリ型（type列など）もここで普通の文字になるので、ルールに縛られなくなります
# export_df = final_all_df.astype(str)

# # 2. そのあとで、NaN（文字になると "<NA>" や "nan" になることがあります）を空文字に変換
# # 文字列になっているので、"nan" という文字を "" に置き換えます
# export_df = export_df.replace(["nan", "NaN", "<NA>", "None"], "")

# # 3. ★重要★ 「列の名前（ヘッダー）」をすべて文字列に変換
# export_df.columns = export_df.columns.astype(str)

# # 4. 書き出し用データの準備
# header = export_df.columns.values.tolist()
# rows = export_df.values.tolist()
# data_to_export = [header] + rows

# # 5. スプレッドシートへの書き出し処理
# SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly',
#           'https://www.googleapis.com/auth/spreadsheets']
# json_path = r"Z:\Users\suehara\Documents\python\analysis\yojitu\python_ss\credentials.json"
# service = ps.get_auth(SCOPES, json_path)

# OUTPUT_SPREADSHEET_ID = '12ws4AVPj6Xu7t0oTDYAmaTN_JpAVCPgk9aYTemJcIl4'
# Sheet_NAME = 'pivot'

# # シートをクリアしてから一気に書き込み
# service.spreadsheets().values().clear(spreadsheetId=OUTPUT_SPREADSHEET_ID, range=Sheet_NAME).execute()
# ps.update_ss(OUTPUT_SPREADSHEET_ID, Sheet_NAME + '!B1', data_to_export, service)

# print("✨ 今度こそ、今度こそ！すべてのデータがスプレッドシートに出力されました！")

In [ ]:
# # ==========================================
# # NaNの原因（リストから漏れている名前）を特定するコード
# # ==========================================

# # 1. 念のため、初期交渉と本交渉のすべての type を「文字」として取得します
# unique_shoki_types = shoki_pivot_df['type'].astype(str).unique()
# unique_hon_types = hon_pivot_df['type'].astype(str).unique()

# # 2. それらをガッチャンコして、存在しているすべての type 名のリストを作ります
# all_existing_types = list(set(unique_shoki_types) | set(unique_hon_types))

# # 3. リストに登録した「full_type_order」と比較して、漏れている犯人を探します
# full_type_order = [
#     'shoki_settei', 'shoki_jisshi',
#     'hon_settei', 'hon_jisshi',
#     'hon_settei_shin', 'hon_settei_sai',
#     'hon_settei_kata', 'hon_settei_kata_shin', 'hon_settei_kata_sai',
#     'hon_settei_ryo', 'hon_settei_ryo_shin', 'hon_settei_ryo_sai',
#     'hon_settei_sha','hon_settei_nin','kigyo_settei','kigyo_settei_sha','kigyo_settei_nin',
    
#     'hon_settei_han',
#     'hon_settei_han_shin','hon_settei_han_sai',
#     'hon_settei_han_kata','hon_settei_han_kata_shin','hon_settei_han_kata_sai',
#     'hon_settei_han_ryo','hon_settei_han_ryo_shin','hon_settei_han_ryo_sai',
#     'hon_settei_han_sha','hon_settei_han_nin',
#     'kigyo_settei_han','kigyo_settei_han_sha','kigyo_settei_han_nin'
# ]

# # 犯人（実際のデータには存在するのに、full_type_order に書いていない名前）をあぶり出します
# missing_types = [t for t in all_existing_types if t not in full_type_order]

# print("🚨 リストに書き忘れている type 名は以下の通りです：")
# print(missing_types)

## 成約数と顧客支持ポイント

In [ ]:
# ==========================================
# 成約数情報の取得（必要な列のみに絞り込み）
# ==========================================

# 1. 取得範囲を AN列まで広げて取得
SPREADSHEET_ID = "1ITzx2eIAMpiepjGVrDSf-FR1XcAKzYJMqFZuS6-8qb0"
Sheet_NAME = 'data!'
Sheet_row = "A:AN" # AN列まで取得するように変更
RANGE_NAME = Sheet_NAME + Sheet_row

# 全データを一旦取得
yomi_full = ps.get_ss(SPREADSHEET_ID, RANGE_NAME, service)

# 2. 必要な列だけを抽出（列名、または列番号で指定）
# ※ps.get_ss が 1行目をヘッダーとして読み込んでいる前提です
target_cols = [
    '案件id（RZ）', '月', '案件No（SC）', '計上日', 'クライアント正式名称', 
    '候補者', '売上種別（商品内容）', '基準年収', '報酬率', 
    '営業売上合計', '所属課', '氏名', '顧客支持ポイント', '貢献引当後pt', 
    '担当', '内定数フラグ'
]

# 存在する列だけを安全に抽出
yomi_old = yomi_full[target_cols].copy()

# 確認表示
display(yomi_old.tail())

,案件id（RZ）,月,案件No（SC）,計上日,クライアント正式名称,候補者,売上種別（商品内容）,基準年収,報酬率,営業売上合計,所属課,氏名,顧客支持ポイント,貢献引当後pt,担当,内定数フラグ
62555,,5,,2026/05/26,株式会社KIKUZAKI電気,,顧問名鑑,,,,ロンザン,ロンザンその他,31.9159,28.7243,None,None
62556,,5,,2026/05/29,日本電力供給株式会社,,顧問名鑑,,,,ロンザン,菊地潤,9.979,8.9811,None,None
62557,,5,,2026/05/29,北光金属株式会社,,顧問名鑑,,,,ロンザン,岸靖,84.983,76.4847,None,None
62558,,6,,2026/06/01,株式会社ギブ・スパイラル・ジャパン,,顧問名鑑,,,,ロンザン,増井真,1.296,1.1664,None,None
62559,,6,,2026/06/01,株式会社ギブ・スパイラル・ジャパン,,顧問名鑑,,,,ロンザン,ロンザンその他,0.864,0.7776,None,None


In [ ]:
# ==========================================
# 今Qのヨミ表を取得（特定のカラムのみ抽出）
# ==========================================

# 1. 取得範囲の設定（AZ列まで広めに取得）
SPREADSHEET_ID = "1yZdENO78p8AkftwaNlwNqP_NQVwKfPYYpjBGjUccflo"
Sheet_NAME = 'ヨミ表!'
Sheet_row = "A10:AZ" # 10行目からAZ列まで
RANGE_NAME = Sheet_NAME + Sheet_row

# 一旦すべてのデータを取得
yomi_nowQ_full = ps.get_ss(SPREADSHEET_ID, RANGE_NAME, service)

# ------------------------------------------
# ★追加：カラム名のクレンジング（ノイズ除去）
# ------------------------------------------
# \s+ は「スペースや改行、タブなどの空白文字」を表します。これを空文字('')に置き換えます。
yomi_nowQ_full.columns = [re.sub(r'\s+', '', str(col)) for col in yomi_nowQ_full.columns]

# 2. 抽出したいカラムのリスト（過去ヨミ表と共通）
target_cols = [
    '案件id（RZ）', '月', '案件No（SC）', '計上日', 'クライアント正式名称', 
    '候補者', '売上種別（商品内容）', '基準年収', '報酬率', 
    '営業売上合計', '所属課', '氏名', '顧客支持ポイント', '貢献引当後pt', 
    '担当', '内定数フラグ'
]

# 3. 指定したカラムだけを抽出し、メモリ効率のためにコピーを作成
# ※列名が存在しない場合のエラーを防ぐため、実際の列に含まれるものだけを抽出します
yomi_nowQ = yomi_nowQ_full[[c for c in target_cols if c in yomi_nowQ_full.columns]].copy()

# 確認表示
display(yomi_nowQ.head())

,案件id（RZ）,月,案件No（SC）,計上日,クライアント正式名称,候補者,売上種別（商品内容）,基準年収,報酬率,営業売上合計,所属課,氏名,顧客支持ポイント,貢献引当後pt,内定数フラグ
1,28413,2026年4月,20402,2026/4/7,株式会社サンマルクホールディングス,飛田 氏,シニアスカウト報酬,900,69%,621.0000,シニア引当,CXL引当,62.1000,55.8900,
2,28413,2026年4月,20402,2026/4/7,株式会社サンマルクホールディングス,飛田 氏,シニアスカウト報酬,900,69%,621.0000,シニア引当,サービス引当,55.8900,50.3010,
3,28413,2026年4月,20402,2026/4/7,株式会社サンマルクホールディングス,飛田 氏,シニアスカウト報酬,900,69%,621.0000,商材間調整,商材間調整,100.6020,90.5418,
4,28413,2026年4月,20402,2026/4/7,株式会社サンマルクホールディングス,飛田 氏,シニアスカウト報酬,900,69%,621.0000,外部原価,外部原価,120.7220,108.6498,
5,28413,2026年4月,20402,2026/4/7,株式会社サンマルクホールディングス,飛田 氏,シニアスカウト報酬,900,69%,621.0000,ロンザン,RZ優良企業開拓引当,8.0280,7.2252,


In [ ]:
yomi_nowQ.columns

Index(['案件id（RZ）', '月', '案件No（SC）', '計上日', 'クライアント正式名称', '候補者', '売上種別（商品内容）',
       '基準年収', '報酬率', '営業売上合計', '所属課', '氏名', '顧客支持ポイント', '貢献引当後pt', '内定数フラグ'],
      dtype='object')

In [ ]:
# ==========================================
# ヨミ表データの縦結合（過去分 + 今Q分）
# ==========================================

# pd.concat を使って縦に結合します
# ignore_index=True を指定することで、インデックス（行番号）を 0 から振り直してキレイにします
yomi_all = pd.concat([yomi_old, yomi_nowQ], ignore_index=True)

yomi_all.loc[yomi_all['氏名'].isin(['京谷悠子', '京谷悠']), '氏名'] = 'yuko-kyotani'
yomi_all.loc[yomi_all['氏名'] == '竹下綾', '氏名'] = 'aya-takeshita'

# 結合結果の確認
print(f"過去データの行数: {len(yomi_old)}")
print(f"今Qデータの行数 : {len(yomi_nowQ)}")
print(f"結合後の合計行数: {len(yomi_all)}")

# データの先頭と末尾を表示して、正しく結合されているか確認
display(yomi_all.head())
display(yomi_all.tail())
display(yomi_all.columns)

過去データの行数: 62559
今Qデータの行数 : 50400
結合後の合計行数: 112959


,案件id（RZ）,月,案件No（SC）,計上日,クライアント正式名称,候補者,売上種別（商品内容）,基準年収,報酬率,営業売上合計,所属課,氏名,顧客支持ポイント,貢献引当後pt,担当,内定数フラグ
0,,4,8606,2016/04/15,株式会社フジダン,白川 正明 氏,スカウト報酬（通常）,630,0.55,,ロンザン,宮崎佳,82.467,82.467,None,None
1,,4,,2016/04/15,株式会社林間,岩本 昌和 氏,スカウト報酬（通常）,740,0.58,,ロンザン,宮崎佳,102.1496,102.1496,None,None
2,内定内定71,4,,2016/04/21,株式会社星野産商,鼠入 宏明 氏,シニアスカウト報酬,1110,0.58,,シニア引当,サービス引当,64.38,64.38,,None
3,内定内定71,4,,2016/04/21,株式会社星野産商,鼠入 宏明 氏,シニアスカウト報酬,1110,0.58,,シニア引当,CXL引当,128.76,128.76,,None
4,内定内定71,4,,2016/04/21,株式会社星野産商,鼠入 宏明 氏,シニアスカウト報酬,1110,0.58,,ロンザン,長崎文,191.5305,191.5305,候補者担当,1


,案件id（RZ）,月,案件No（SC）,計上日,クライアント正式名称,候補者,売上種別（商品内容）,基準年収,報酬率,営業売上合計,所属課,氏名,顧客支持ポイント,貢献引当後pt,担当,内定数フラグ
112954,,,None,,,,,,,,,,,0.0000,NaN,None
112955,,,None,,,,,,,,,,,0.0000,NaN,None
112956,,,None,,,,,,,,,,,0.0000,NaN,None
112957,,,None,,,,,,,,,,,0.0000,NaN,None
112958,,,None,,,,,,,,,,,0.0000,NaN,None


Index(['案件id（RZ）', '月', '案件No（SC）', '計上日', 'クライアント正式名称', '候補者', '売上種別（商品内容）',
       '基準年収', '報酬率', '営業売上合計', '所属課', '氏名', '顧客支持ポイント', '貢献引当後pt', '担当',
       '内定数フラグ'],
      dtype='object')

In [ ]:
# ==========================================
# yomi_all の前処理とマスタマージ
# ==========================================

# 1. データ型の変換
# '計上日' を日付型に変換
yomi_all['計上日'] = pd.to_datetime(yomi_all['計上日'], errors='coerce')

# それ以外の列を object 型に一括変換
other_cols = yomi_all.columns.difference(['計上日'])
yomi_all[other_cols] = yomi_all[other_cols].astype('object')


# 2. Q_masterのマージ（計上日基準）
yomi_all = (
    yomi_all.merge(q_master_subset, left_on='計上日', right_on='日付', how='left')
    .drop(columns=['日付'])
    .rename(columns={
        'Q': 'keijo_Q', 
        '月': 'keijo_月', 
        '営業日': 'keijo_営業日', 
        '同営業日比較': 'keijo_同営業日比較'
    })
)


In [ ]:
# 3. honkosho_data から「担当者属性マスター」を作成
# 案件IDに紐づく担当者や属性は基本的に1つのため、重複を排除してマスター化します
hon_tanto_master = honkosho_data[[
    'anken_id', 'kohosha_id', 'APsource', 
    'kohosha_tanto', 'kigyo_tanto', 
    '候担_職種', '企担_職種', 
    '候担_レイヤー', '企担_レイヤー', 
    '候担_ロンザン所属フラグ', '企担_ロンザン所属フラグ', '組み手', 'hanjokin'
]].drop_duplicates('anken_id')


In [ ]:
# ==========================================
# マージ前の徹底洗浄（ここが重要！）
# ==========================================

# 1. hon_tanto_master のキーを洗浄
hon_tanto_master['anken_id'] = hon_tanto_master['anken_id'].astype(str).str.strip()

# 2. yomi_all のカラム名を扱いやすくリネーム（改行コード対策）
yomi_all = yomi_all.rename(columns={c: '案件id_RZ' for c in yomi_all.columns if '案件id' in c})
yomi_all = yomi_all.rename(columns={c: '案件No_SC' for c in yomi_all.columns if '案件' in c and 'No' in c})

# 3. yomi_all のキーを洗浄（数値型を文字列に変え、空白を消す）
yomi_all['案件id_RZ'] = yomi_all['案件id_RZ'].astype(str).str.strip()

# 4. すでに yomi_all に存在する「マージで持ってきたい列」を一旦削除（重複防止）
# これをしないと _x _y という列が増えるだけで中身が NaN のままに見えます
cols_to_drop = [
    'kohosha_id', 'APsource', 'kohosha_tanto', 'kigyo_tanto', 
    '候担_職種', '企担_職種', '候担_レイヤー', '企担_レイヤー', 
    '候担_ロンザン所属フラグ', '企担_ロンザン所属フラグ', '組み手'
]
yomi_all = yomi_all.drop(columns=[c for c in cols_to_drop if c in yomi_all.columns])

# ==========================================
# いざ、再マージ！
# ==========================================
yomi_all = yomi_all.merge(
    hon_tanto_master, 
    left_on='案件id_RZ', 
    right_on='anken_id', 
    how='left'
)

# 洗浄のために作った temporary な列を消してスッキリ
if 'anken_id' in yomi_all.columns:
    yomi_all = yomi_all.drop(columns=['anken_id'])



In [ ]:

# 結果確認
print("紐付け成功数（NaNでない数）:", yomi_all['kohosha_id'].notna().sum())
display(yomi_all.sample(min(10, len(yomi_all))))

# ==========================================
# ローデータ（yomi_all）のスプレッドシート出力
# ==========================================

# 1. 出力用データの準備
output_raw_df = yomi_all.copy()

# 【エラー対策】全ての datetime/Timestamp 型の列を文字列に変換します
# これにより JSON serializable な形式になります
for col in output_raw_df.select_dtypes(include=['datetime', 'datetime64']).columns:
    # 日付形式（YYYY-MM-DD）の文字列に変換。時間が不要な場合は strftime('%Y-%m-%d')
    output_raw_df[col] = output_raw_df[col].dt.strftime('%Y-%m-%d')

# 欠損値を空文字に変換（NaT も上の処理で NaN になっている場合はここで空文字になります）
output_raw_df = output_raw_df.fillna("")

# 2. DataFrameをヘッダー付きのリスト形式に変換
header = output_raw_df.columns.tolist()
values_to_send = [header] + output_raw_df.values.tolist()

# 3. 書き出し先の設定
RAW_SPREADSHEET_ID = "12ws4AVPj6Xu7t0oTDYAmaTN_JpAVCPgk9aYTemJcIl4"
RAW_SHEET_NAME = "raw_seiyaku"

print(f"「{RAW_SHEET_NAME}」シートへの書き出しを開始します...")

# 4. 指定したシートのA1セルから書き出しを実行
# 既存のヘルパー関数 ps.update_ss を使用します
ps.update_ss(RAW_SPREADSHEET_ID, f"{RAW_SHEET_NAME}!A1", values_to_send, service)

print(f"✅ スプレッドシートへの出力が完了しました！")

紐付け成功数（NaNでない数）: 54656


,案件id_RZ,月_x,案件No_SC,計上日,クライアント正式名称,候補者,売上種別（商品内容）,基準年収,報酬率,営業売上合計,所属課,氏名,顧客支持ポイント,貢献引当後pt,担当,内定数フラグ,keijo_Q,月_y,keijo_営業日,keijo_同営業日比較,kohosha_id,APsource,kohosha_tanto,kigyo_tanto,候担_職種,企担_職種,候担_レイヤー,企担_レイヤー,候担_ロンザン所属フラグ,企担_ロンザン所属フラグ,組み手,hanjokin
12308,5990,,10952,2020-02-19,株式会社富士薬品,清水 陽一 氏,シニアスカウト報酬,1100,0.58,638,シニア引当,サービス引当,63.8,63.8,CXL引当①,None,23-2Q,2020-02-01,32,同営業日,14793,その他,増田智,増田智,ミドル候B,ミドル候B,中途,中途,1,1,両手,
90634,,,None,NaT,,,,,,,,,,0.0000,NaN,None,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
108020,,,None,NaT,,,,,,,,,,0.0000,NaN,None,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
85646,,,None,NaT,,,,,,,,,,0.0000,NaN,None,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36986,17135,10,6762,2023-10-23,ユーハ株式会社,中川 雅史 氏,シニアスカウト報酬,850,0.48,408,ロンザン,大仲研,13.643,12.27894474,営業・設定担当,None,27-1Q,2023-10-01,12,同営業日,46847,SMAP,前田崚,大仲研,ミドル候A,フロント,中途：26期後期,部責,1,1,両手,
21533,11019,,11473,2021-06-18,株式会社イシグロ,安藤 和彦 氏,シニアスカウト報酬,1140,0.58,661.2,SC,播本大,13.850487,13.850487,他部署現S担当,None,24-3Q,2021-06-01,52,,24091,SMAP,三浪純,菊地潤,ミドル候B,ミドル候B,中途,中途：23期後期,1,1,両手,
32237,15494,2,11495,2023-02-28,大成ファインケミカル株式会社,黒川 聡 氏,シニアスカウト報酬,1430,1,1430,ロンザン,その他（候）,12.55231177,12.55231177,,None,26-2Q,2023-02-01,38,同営業日,40085,人事部紹介,大塚洋,笹原啓,SMAP,フロント,既存,中途：22期後期,1,1,両手,
75177,,,None,NaT,,,,,,,,,,0.0000,NaN,None,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31102,15067,,11688,2022-12-27,髙橋金属株式会社,柘植 浩二 氏,シニアスカウト報酬,1847.92,0.65,1201.148,シニア引当,CXL引当,120.1148,120.1148,CXL引当,None,26-1Q,2022-12-01,57,,39166,SMAP,木村真２,角田隆,ミドル候B,ミドル企業,021生,既存,1,1,両手,
66127,,,None,NaT,,,,,,,,,,0.0000,NaN,None,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


「raw_seiyaku」シートへの書き出しを開始します...
✅ スプレッドシートへの出力が完了しました！


In [ ]:
# ==========================================
# セル: 成約・ヨミ表データの詳細抽出と属性紐付け（修正版）
# ==========================================

# 1. 準備：最新の担当者属性を「計上Q + 氏名」で紐付け直す
yomi_calc_base = yomi_all.copy()

# 紐付け用に master2 を準備（重複削除）
attr_master = master2[['Q', 'sei_plus', '職種', 'レイヤー', 'ロンザン所属フラグ']].drop_duplicates(['Q', 'sei_plus'])

# マージ「する前」に attr_master 側のカラム名を変更しておきます
attr_master = attr_master.rename(columns={
    '職種': '集計用_職種', 
    'レイヤー': '集計用_レイヤー', 
    'ロンザン所属フラグ': '集計用_所属フラグ'
})

# マージ実行
yomi_calc_base = yomi_calc_base.merge(
    attr_master, 
    left_on=['keijo_Q', '氏名'], 
    right_on=['Q', 'sei_plus'], 
    how='left'
)

# 2. 抽出用マスクの定義とデータクレンジング
def is_blank(series):
    s = series.astype(str).str.strip().str.lower()
    return series.isna() | s.isin(['', 'nan', 'none', 'null', '0', '0.0'])

# ★追加：企業成約社数用のクレンジング（「株式会社」やスペースを削除）
yomi_calc_base['クライアント正式名称_クレンジング'] = (
    yomi_calc_base['クライアント正式名称']
    .astype(str)
    .str.replace(r'株式会社|（株）|\(株\)', '', regex=True)
    .str.replace(r'\s+', '', regex=True)
)

mask_ronzan = yomi_calc_base['所属課'].astype(str).str.contains('ロンザン', na=False)
mask_kotei = yomi_calc_base['売上種別（商品内容）'].astype(str).str.contains('固定報酬', na=False)
mask_blank_rz = is_blank(yomi_calc_base['案件id_RZ'])
mask_not_blank = (~mask_blank_rz) 


# ==========================================
# ★変更：用途に応じた独立マスクの作成（ベクタライズ処理）
# ==========================================
# 【案件ベース】（プランで判定）
mask_anken_han = yomi_calc_base['hanjokin'] == '半常勤'

# 【人ベース】（担当者・氏名で判定）
# ※欠損値を考慮し、文字列に変換して部分一致を判定
mask_koho_han  = yomi_calc_base['候担_職種'].astype(str).str.contains('半常勤')
mask_kigyo_han = yomi_calc_base['企担_職種'].astype(str).str.contains('半常勤')
mask_point_han = yomi_calc_base['集計用_職種'].astype(str).str.contains('半常勤')


# --- A. 成約数系マスク ---
mask_naitei = yomi_calc_base['内定数フラグ'].astype(str).str.strip().str.startswith('1')
mask_koho_rz = yomi_calc_base['候担_ロンザン所属フラグ'].astype(str).str.contains('1')
mask_kigyo_rz = yomi_calc_base['企担_ロンザン所属フラグ'].astype(str).str.contains('1')
mask_kata = yomi_calc_base['組み手'] == '片手'
mask_ryo = yomi_calc_base['組み手'] == '両手'


# 3. 各 type ごとのデータ抽出とリスト格納
seiyaku_dfs = []

# ==========================================
# 【常勤メンバー / 常勤案件】（~mask_xxx_han で否定して抽出）
# ==========================================

# --- A. 案件ベースでの判定 ---
# 成約数（★新規type）
seiyaku_dfs.append(yomi_calc_base[mask_naitei & ~mask_anken_han].assign(type='seiyaku_anken'))
# 顧客支持ポイント（★新規type）
seiyaku_dfs.append(yomi_calc_base[mask_ronzan & mask_not_blank & ~mask_kotei & ~mask_anken_han].assign(type='point_anken'))
# 両手成約（案件ベースで判定）
seiyaku_dfs.append(yomi_calc_base[mask_naitei & mask_ryo & ~mask_anken_han].assign(type='seiyaku_ryo'))

# --- B. 人ベースでの判定 ---
# 候補者成約 (候補者担当で判定)
seiyaku_dfs.append(yomi_calc_base[mask_naitei & mask_koho_rz & ~mask_koho_han].assign(type='seiyaku'))
seiyaku_dfs.append(yomi_calc_base[mask_naitei & mask_kata & ~mask_koho_han].assign(type='seiyaku_kata'))

# 企業成約 (企業担当で判定)
seiyaku_dfs.append(yomi_calc_base[mask_naitei & mask_kigyo_rz & ~mask_kigyo_han].assign(type='kigyo_seiyaku'))
seiyaku_dfs.append(yomi_calc_base[mask_naitei & mask_kigyo_rz & ~mask_kigyo_han].assign(type='kigyo_seiyaku_nin'))
seiyaku_dfs.append(yomi_calc_base[mask_naitei & mask_kigyo_rz & ~mask_kigyo_han].assign(type='kigyo_seiyaku_sha'))

# 顧客支持ポイント (氏名で判定)
seiyaku_dfs.append(yomi_calc_base[mask_ronzan & mask_not_blank & ~mask_kotei & ~mask_point_han].assign(type='point'))
seiyaku_dfs.append(yomi_calc_base[mask_ronzan & mask_not_blank & ~mask_kotei & mask_kata & ~mask_point_han].assign(type='point_kata'))
seiyaku_dfs.append(yomi_calc_base[mask_ronzan & mask_not_blank & ~mask_kotei & mask_ryo & ~mask_point_han].assign(type='point_ryo'))
seiyaku_dfs.append(yomi_calc_base[mask_ronzan & mask_blank_rz & ~mask_point_han].assign(type='point_cross'))
seiyaku_dfs.append(yomi_calc_base[mask_ronzan & mask_not_blank & mask_kotei & ~mask_point_han].assign(type='point_kotei'))


# ==========================================
# 【半常勤メンバー / 半常勤案件】
# ==========================================

# --- A. 案件ベースでの判定 ---
seiyaku_dfs.append(yomi_calc_base[mask_naitei & mask_anken_han].assign(type='seiyaku_anken_han'))
seiyaku_dfs.append(yomi_calc_base[mask_ronzan & mask_not_blank & ~mask_kotei & mask_anken_han].assign(type='point_anken_han'))
seiyaku_dfs.append(yomi_calc_base[mask_naitei & mask_ryo & mask_anken_han].assign(type='seiyaku_ryo_han'))

# --- B. 人ベースでの判定 ---
# 候補者成約
seiyaku_dfs.append(yomi_calc_base[mask_naitei & mask_koho_rz & mask_koho_han].assign(type='seiyaku_han'))
seiyaku_dfs.append(yomi_calc_base[mask_naitei & mask_kata & mask_koho_han].assign(type='seiyaku_kata_han'))

# 企業成約
seiyaku_dfs.append(yomi_calc_base[mask_naitei & mask_kigyo_rz & mask_kigyo_han].assign(type='kigyo_seiyaku_han'))
seiyaku_dfs.append(yomi_calc_base[mask_naitei & mask_kigyo_rz & mask_kigyo_han].assign(type='kigyo_seiyaku_nin_han'))
seiyaku_dfs.append(yomi_calc_base[mask_naitei & mask_kigyo_rz & mask_kigyo_han].assign(type='kigyo_seiyaku_sha_han'))

# 顧客支持ポイント
seiyaku_dfs.append(yomi_calc_base[mask_ronzan & mask_not_blank & ~mask_kotei & mask_point_han].assign(type='point_han'))
seiyaku_dfs.append(yomi_calc_base[mask_ronzan & mask_blank_rz & mask_point_han].assign(type='point_cross_han'))
seiyaku_dfs.append(yomi_calc_base[mask_ronzan & mask_not_blank & mask_kotei & mask_point_han].assign(type='point_kotei_han'))

# 縦積み
seiyaku_combined_data = pd.concat(seiyaku_dfs, ignore_index=True)

# 顧客支持ポイントと貢献引当後ptを数値化
seiyaku_combined_data['顧客支持ポイント'] = pd.to_numeric(seiyaku_combined_data['顧客支持ポイント'], errors='coerce').fillna(0)
seiyaku_combined_data['貢献引当後pt'] = pd.to_numeric(seiyaku_combined_data['貢献引当後pt'], errors='coerce').fillna(0)

# 結果確認
print("結合後のデータの行数と列数:", seiyaku_combined_data.shape)
print("\n▼ 作成された type ごとの件数")
print(seiyaku_combined_data['type'].value_counts())

結合後のデータの行数と列数: (58283, 39)

▼ 作成された type ごとの件数
type
point                    20536
point_ryo                12109
point_kata                7962
point_cross               5072
seiyaku                   3523
seiyaku_kata              1953
kigyo_seiyaku_nin         1654
kigyo_seiyaku             1654
kigyo_seiyaku_sha         1654
seiyaku_ryo               1612
point_kotei                201
point_han                  147
point_cross_han             97
point_kotei_han             26
seiyaku_han                 19
seiyaku_ryo_han             15
kigyo_seiyaku_han           15
kigyo_seiyaku_nin_han       15
kigyo_seiyaku_sha_han       15
seiyaku_kata_han             4
Name: count, dtype: int64


In [ ]:
print(seiyaku_combined_data.columns.tolist())

['案件id_RZ', '月_x', '案件No_SC', '計上日', 'クライアント正式名称', '候補者', '売上種別（商品内容）', '基準年収', '報酬率', '営業売上合計', '所属課', '氏名', '顧客支持ポイント', '貢献引当後pt', '担当', '内定数フラグ', 'keijo_Q', '月_y', 'keijo_営業日', 'keijo_同営業日比較', 'kohosha_id', 'APsource', 'kohosha_tanto', 'kigyo_tanto', '候担_職種', '企担_職種', '候担_レイヤー', '企担_レイヤー', '候担_ロンザン所属フラグ', '企担_ロンザン所属フラグ', '組み手', 'hanjokin', 'Q', 'sei_plus', '集計用_職種', '集計用_レイヤー', '集計用_所属フラグ', 'クライアント正式名称_クレンジング', 'type']


In [ ]:
# ==========================================
# セル: 成約・ヨミ表のピボットテーブル作成（詳細版・高速化）
# ==========================================

def aggregate_seiyaku_data(df: pd.DataFrame, valid_members: list, type_order: list) -> pd.DataFrame:
    df = df.copy(deep=False)
    
    # 1. 派生カラムの作成
    df['keijo_Q_同営業日'] = df['keijo_Q'].astype(str) + "同営業日"
    df['全体'] = '合計' # ★ここを追加
    if 'value' not in df.columns:
        df['value'] = 1
    
    time_columns = ['keijo_Q', '月_y', 'keijo_Q_同営業日']
    time_axis_dfs = []
    
    for time_col in time_columns:
        pivot_results = []
        
        for t_name in type_order:
            df_sub = df[df['type'] == t_name]
            if df_sub.empty:
                continue
            
            is_point = 'point' in t_name
            is_nin = 'nin' in t_name
            is_sha = 'sha' in t_name
            
            # --- 集計関数の決定 ---
            if is_nin or is_sha:
                agg_func = 'nunique'
            else:
                agg_func = 'sum'
            
            # --- 対象カラムの決定 ---
            if 'kigyo_seiyaku' in t_name:
                job_col, layer_col, tanto_col = '企担_職種', '企担_レイヤー', 'kigyo_tanto'
            elif is_point:
                job_col, layer_col, tanto_col = '集計用_職種', '集計用_レイヤー', '氏名'
            else: 
                job_col, layer_col, tanto_col = '候担_職種', '候担_レイヤー', 'kohosha_tanto'

            # --- 集計値(values)の決定 ---
            if is_point:
                val_col_ap = '顧客支持ポイント'
                val_col_job = '貢献引当後pt'
            elif is_nin:
                val_col_ap = val_col_job = '候補者'
            elif is_sha:
                val_col_ap = val_col_job = 'クライアント正式名称_クレンジング'
            else:
                val_col_ap = val_col_job = 'value'

            # --- ピボット作成 ---
            # ★合計（全体）の集計を追加
            p_total = df_sub.pivot_table(index=['type', '全体'], columns=time_col, values=val_col_ap, aggfunc=agg_func).fillna(0).reset_index()
            pivot_results.append(p_total.rename(columns={'全体': '項目名'}).assign(集計軸='全体'))

            p_ap = df_sub.pivot_table(index=['type', 'APsource'], columns=time_col, values=val_col_ap, aggfunc=agg_func).fillna(0).reset_index()
            pivot_results.append(p_ap.rename(columns={'APsource': '項目名'}).assign(集計軸='APソース'))

            p_job = df_sub.pivot_table(index=['type', job_col], columns=time_col, values=val_col_job, aggfunc=agg_func).fillna(0).reset_index()
            pivot_results.append(p_job.rename(columns={job_col: '項目名'}).assign(集計軸='職種'))

            p_layer = df_sub.pivot_table(index=['type', layer_col], columns=time_col, values=val_col_job, aggfunc=agg_func).fillna(0).reset_index()
            pivot_results.append(p_layer.rename(columns={layer_col: '項目名'}).assign(集計軸='レイヤー'))

            df_tanto = df_sub[df_sub[tanto_col].isin(valid_members)]
            p_tanto = df_tanto.pivot_table(index=['type', tanto_col], columns=time_col, values=val_col_job, aggfunc=agg_func).fillna(0).reset_index()
            pivot_results.append(p_tanto.rename(columns={tanto_col: '項目名'}).assign(集計軸='担当'))
            
        if pivot_results:
            combined_axis = pd.concat(pivot_results, ignore_index=True)
            combined_axis = combined_axis.set_index(["type", "集計軸", "項目名"])
            time_axis_dfs.append(combined_axis)

    final_df = pd.concat(time_axis_dfs, axis=1).fillna(0).reset_index()
    final_df['type'] = pd.Categorical(final_df['type'], categories=type_order, ordered=True)
    
    numeric_cols = final_df.columns.difference(['type', '集計軸', '項目名'])
    mask_not_zero = final_df[numeric_cols].sum(axis=1) > 0
    mask_keep_zero = final_df['type'].astype(str).str.contains('point_cross')
    final_df = final_df[mask_not_zero | mask_keep_zero]
    
    final_df = final_df.sort_values(by=['type', '集計軸', '項目名']).reset_index(drop=True)
    final_df.columns.name = None
    
    return final_df

# --- 実行部分 ---
seiyaku_types = [
    'seiyaku', 'seiyaku_kata', 'seiyaku_ryo',
    'seiyaku_han', 'seiyaku_kata_han', 'seiyaku_ryo_han',
    'kigyo_seiyaku', 'kigyo_seiyaku_nin', 'kigyo_seiyaku_sha',
    'kigyo_seiyaku_han', 'kigyo_seiyaku_nin_han', 'kigyo_seiyaku_sha_han',
    'point', 'point_kata', 'point_ryo', 'point_cross', 'point_kotei',
    'point_han', 'point_cross_han', 'point_kotei_han',
    # ★新規追加分
    'seiyaku_anken', 'point_anken', 'seiyaku_anken_han', 'point_anken_han',
]

valid_members = master2['sei_plus'].dropna().unique()

# 関数の実行
seiyaku_pivot_df = aggregate_seiyaku_data(seiyaku_combined_data, valid_members, seiyaku_types)

# 確認
print("【成約・ヨミ表データの新しい集計テーブル】")
display(seiyaku_pivot_df.head(10))

【成約・ヨミ表データの新しい集計テーブル】


,type,集計軸,項目名,19-4Q,20-1Q,20-2Q,20-3Q,20-4Q,21-1Q,21-2Q,21-3Q,21-4Q,22-1Q,22-2Q,22-3Q,22-4Q,23-1Q,23-2Q,23-3Q,23-4Q,24-1Q,24-2Q,24-3Q,24-4Q,25-1Q,25-2Q,25-3Q,25-4Q,26-1Q,26-2Q,26-3Q,26-4Q,27-1Q,27-2Q,27-3Q,27-4Q,28-1Q,28-2Q,28-3Q,28-4Q,29-1Q,29-2Q,29-3Q,19-3Q,2016-08-01 00:00:00,2016-09-01 00:00:00,2016-10-01 00:00:00,2016-11-01 00:00:00,2016-12-01 00:00:00,2017-01-01 00:00:00,2017-02-01 00:00:00,2017-03-01 00:00:00,2017-04-01 00:00:00,2017-05-01 00:00:00,2017-06-01 00:00:00,2017-07-01 00:00:00,2017-08-01 00:00:00,2017-09-01 00:00:00,2017-10-01 00:00:00,2017-11-01 00:00:00,2017-12-01 00:00:00,2018-01-01 00:00:00,2018-02-01 00:00:00,2018-03-01 00:00:00,2018-04-01 00:00:00,2018-05-01 00:00:00,2018-06-01 00:00:00,2018-07-01 00:00:00,2018-08-01 00:00:00,2018-09-01 00:00:00,2018-10-01 00:00:00,2018-11-01 00:00:00,2018-12-01 00:00:00,2019-01-01 00:00:00,2019-02-01 00:00:00,2019-03-01 00:00:00,2019-04-01 00:00:00,2019-05-01 00:00:00,2019-06-01 00:00:00,2019-07-01 00:00:00,2019-08-01 00:00:00,2019-09-01 00:00:00,2019-10-01 00:00:00,2019-11-01 00:00:00,2019-12-01 00:00:00,2020-01-01 00:00:00,2020-02-01 00:00:00,2020-03-01 00:00:00,2020-04-01 00:00:00,2020-05-01 00:00:00,2020-06-01 00:00:00,2020-07-01 00:00:00,2020-08-01 00:00:00,2020-09-01 00:00:00,2020-10-01 00:00:00,2020-11-01 00:00:00,2020-12-01 00:00:00,2021-01-01 00:00:00,2021-02-01 00:00:00,2021-03-01 00:00:00,...,2021-12-01 00:00:00,2022-01-01 00:00:00,2022-02-01 00:00:00,2022-03-01 00:00:00,2022-04-01 00:00:00,2022-05-01 00:00:00,2022-06-01 00:00:00,2022-07-01 00:00:00,2022-08-01 00:00:00,2022-09-01 00:00:00,2022-10-01 00:00:00,2022-11-01 00:00:00,2022-12-01 00:00:00,2023-01-01 00:00:00,2023-02-01 00:00:00,2023-03-01 00:00:00,2023-04-01 00:00:00,2023-05-01 00:00:00,2023-06-01 00:00:00,2023-07-01 00:00:00,2023-08-01 00:00:00,2023-09-01 00:00:00,2023-10-01 00:00:00,2023-11-01 00:00:00,2023-12-01 00:00:00,2024-01-01 00:00:00,2024-02-01 00:00:00,2024-03-01 00:00:00,2024-04-01 00:00:00,2024-05-01 00:00:00,2024-06-01 00:00:00,2024-07-01 00:00:00,2024-08-01 00:00:00,2024-09-01 00:00:00,2024-10-01 00:00:00,2024-11-01 00:00:00,2024-12-01 00:00:00,2025-01-01 00:00:00,2025-02-01 00:00:00,2025-03-01 00:00:00,2025-04-01 00:00:00,2025-05-01 00:00:00,2025-06-01 00:00:00,2025-07-01 00:00:00,2025-08-01 00:00:00,2025-09-01 00:00:00,2025-10-01 00:00:00,2025-11-01 00:00:00,2025-12-01 00:00:00,2026-01-01 00:00:00,2026-02-01 00:00:00,2026-03-01 00:00:00,2026-04-01 00:00:00,2026-05-01 00:00:00,2026-06-01 00:00:00,2016-04-01 00:00:00,2016-05-01 00:00:00,2016-06-01 00:00:00,2016-07-01 00:00:00,19-4Q同営業日,20-1Q同営業日,20-2Q同営業日,20-3Q同営業日,20-4Q同営業日,21-1Q同営業日,21-2Q同営業日,21-3Q同営業日,21-4Q同営業日,22-1Q同営業日,22-2Q同営業日,22-3Q同営業日,22-4Q同営業日,23-1Q同営業日,23-2Q同営業日,23-3Q同営業日,23-4Q同営業日,24-1Q同営業日,24-2Q同営業日,24-3Q同営業日,24-4Q同営業日,25-1Q同営業日,25-2Q同営業日,25-3Q同営業日,25-4Q同営業日,26-1Q同営業日,26-2Q同営業日,26-3Q同営業日,26-4Q同営業日,27-1Q同営業日,27-2Q同営業日,27-3Q同営業日,27-4Q同営業日,28-1Q同営業日,28-2Q同営業日,28-3Q同営業日,28-4Q同営業日,29-1Q同営業日,29-2Q同営業日,29-3Q同営業日,19-3Q同営業日
0,seiyaku,APソース,SMAP,4.0,8.0,4.0,4.0,8.0,4.0,2.0,4.0,5.0,5.0,9.0,13.0,12.0,6.0,17.0,8.0,15.0,12.0,22.0,16.0,14.0,15.0,12.0,12.0,17.0,20.0,32.0,31.0,60.0,46.0,52.0,46.0,78.0,41.0,54.0,84.0,77.0,60.0,79.0,39.0,0.0,2.0,2.0,2.0,5.0,1.0,2.0,1.0,1.0,1.0,1.0,2.0,2.0,3.0,3.0,0.0,2.0,2.0,0.0,0.0,2.0,2.0,0.0,2.0,0.0,1.0,4.0,0.0,2.0,3.0,2.0,2.0,5.0,2.0,5.0,6.0,4.0,4.0,4.0,2.0,1.0,3.0,2.0,5.0,10.0,1.0,0.0,7.0,2.0,6.0,7.0,2.0,1.0,9.0,7.0,2.0,13.0,...,8.0,1.0,4.0,7.0,4.0,2.0,6.0,6.0,1.0,10.0,5.0,5.0,10.0,7.0,9.0,16.0,9.0,7.0,16.0,16.0,15.0,28.0,9.0,10.0,27.0,7.0,20.0,25.0,3.0,17.0,26.0,25.0,16.0,37.0,11.0,16.0,14.0,16.0,15.0,23.0,18.0,22.0,44.0,20.0,18.0,39.0,15.0,16.0,29.0,17.0,17.0,45.0,16.0,15.0,8.0,0.0,0.0,0.0,0.0,4.0,8.0,4.0,4.0,8.0,4.0,2.0,4.0,5.0,5.0,9.0,13.0,12.0,6.0,17.0,8.0,15.0,12.0,22.0,16.0,14.0,15.0,12.0,12.0,17.0,20.0,32.0,31.0,60.0,46.0,52.0,46.0,78.0,41.0,54.0,84.0,77.0,60.0,79.0,39.0,0.0
1,seiyaku,APソース,その他,1.0,3.0,6.0,5.0,5.0,5.0,9.0,6.0,4.0,5.0,2.0,4.0,1.0,0.0,5.0,3.0,7.0,4.0,7.0,7.0,4.0,2.0,5.0,0.0,2.0,1

In [ ]:
# ==========================================
# 最終工程: データの統合とスプレッドシート書き出し
# ==========================================

# 1. 各データの type を文字列型に揃える
shoki_pivot_df['type'] = shoki_pivot_df['type'].astype(str)
hon_pivot_df['type'] = hon_pivot_df['type'].astype(str)
seiyaku_pivot_df['type'] = seiyaku_pivot_df['type'].astype(str)

# 2. 3つのデータを縦にガッチャンコ！
final_all_df = pd.concat([shoki_pivot_df, hon_pivot_df, seiyaku_pivot_df], ignore_index=True)

# 3. 全KPI（type）の最終的な並び順リストを定義
full_type_order = [
    'shoki_settei', 'shoki_jisshi',
    'hon_settei', 'hon_jisshi',
    'hon_settei_shin', 'hon_settei_sai',
    'hon_settei_kata', 'hon_settei_kata_shin', 'hon_settei_kata_sai',
    'hon_settei_ryo', 'hon_settei_ryo_shin', 'hon_settei_ryo_sai',
    'hon_settei_sha', 'hon_settei_nin',
    'kigyo_settei', 'kigyo_settei_sha', 'kigyo_settei_nin',
    
    'hon_settei_han',
    'hon_settei_han_shin','hon_settei_han_sai',
    'hon_settei_han_kata','hon_settei_han_kata_shin','hon_settei_han_kata_sai',
    'hon_settei_han_ryo','hon_settei_han_ryo_shin','hon_settei_han_ryo_sai',
    'hon_settei_han_sha','hon_settei_han_nin',
    'kigyo_settei_han','kigyo_settei_han_sha','kigyo_settei_han_nin',
    
    # 新規追加の成約系20種
    'seiyaku', 'seiyaku_kata', 'seiyaku_ryo',
    'seiyaku_han', 'seiyaku_kata_han', 'seiyaku_ryo_han',
    'kigyo_seiyaku', 'kigyo_seiyaku_nin', 'kigyo_seiyaku_sha',
    'kigyo_seiyaku_han', 'kigyo_seiyaku_nin_han', 'kigyo_seiyaku_sha_han',
    'point', 'point_kata', 'point_ryo', 'point_cross', 'point_kotei',
    'point_han', 'point_cross_han', 'point_kotei_han',
    # ★新規追加の案件ベース成約・ポイント
    'seiyaku_anken', 'seiyaku_anken_han',
    'point_anken', 'point_anken_han'
]

# 4. カテゴリ型を適用して、理想の順番でソート
final_all_df['type'] = pd.Categorical(final_all_df['type'], categories=full_type_order, ordered=True)

numeric_cols = final_all_df.columns.difference(['type', '集計軸', '項目名'])
final_all_df[numeric_cols] = final_all_df[numeric_cols].fillna(0)

# 並び替え実行
final_all_df = final_all_df.sort_values(by=['type', '集計軸', '項目名'])
final_all_df.columns.name = None


# ==========================================
# Googleスプレッドシートへの書き出し
# ==========================================
print("最終データの書き出しを開始します...")

# 究極の文字列化（JSONエラーを完全に防ぐ）
export_output_df = final_all_df.astype(str)
export_output_df = export_output_df.replace(["nan", "NaN", "<NA>", "None", "NaT"], "")
export_output_df.columns = export_output_df.columns.astype(str)

values_to_send = [export_output_df.columns.tolist()] + export_output_df.values.tolist()

RESULT_SPREADSHEET_ID = "12ws4AVPj6Xu7t0oTDYAmaTN_JpAVCPgk9aYTemJcIl4"
SHEET_NAME = 'pivot'

# シートをクリアしてから一気に書き込み
service.spreadsheets().values().clear(spreadsheetId=RESULT_SPREADSHEET_ID, range=f"{SHEET_NAME}!B:ZZ").execute()
ps.update_ss(RESULT_SPREADSHEET_ID, f"{SHEET_NAME}!B1", values_to_send, service)

print("✅ 全データの統合とスプレッドシートへの出力が完了しました！")
display(final_all_df.head())

最終データの書き出しを開始します...
✅ 全データの統合とスプレッドシートへの出力が完了しました！


,type,集計軸,項目名,18-1Q,18-2Q,18-3Q,18-4Q,19-1Q,19-2Q,19-3Q,19-4Q,20-1Q,20-2Q,20-3Q,20-4Q,21-1Q,21-2Q,21-3Q,21-4Q,22-1Q,22-2Q,22-3Q,22-4Q,23-1Q,23-2Q,23-3Q,23-4Q,24-1Q,24-2Q,24-3Q,24-4Q,25-1Q,25-2Q,25-3Q,25-4Q,26-1Q,26-2Q,26-3Q,26-4Q,27-1Q,27-2Q,27-3Q,27-4Q,28-1Q,28-2Q,28-3Q,28-4Q,29-1Q,29-2Q,29-3Q,2014-11-01 00:00:00,2014-12-01 00:00:00,2015-01-01 00:00:00,2015-02-01 00:00:00,2015-03-01 00:00:00,2015-04-01 00:00:00,2015-07-01 00:00:00,2015-10-01 00:00:00,2015-11-01 00:00:00,2015-12-01 00:00:00,2016-01-01 00:00:00,2016-02-01 00:00:00,2016-03-01 00:00:00,2016-04-01 00:00:00,2016-05-01 00:00:00,2016-06-01 00:00:00,2016-07-01 00:00:00,2016-08-01 00:00:00,2016-09-01 00:00:00,2016-10-01 00:00:00,2016-11-01 00:00:00,2016-12-01 00:00:00,2017-01-01 00:00:00,2017-02-01 00:00:00,2017-03-01 00:00:00,2017-04-01 00:00:00,2017-05-01 00:00:00,2017-06-01 00:00:00,2017-07-01 00:00:00,2017-08-01 00:00:00,2017-09-01 00:00:00,2017-10-01 00:00:00,2017-11-01 00:00:00,2017-12-01 00:00:00,2018-01-01 00:00:00,2018-02-01 00:00:00,2018-03-01 00:00:00,2018-04-01 00:00:00,2018-05-01 00:00:00,2018-06-01 00:00:00,2018-07-01 00:00:00,2018-08-01 00:00:00,2018-09-01 00:00:00,2018-10-01 00:00:00,2018-11-01 00:00:00,2018-12-01 00:00:00,2019-01-01 00:00:00,2019-02-01 00:00:00,2019-03-01 00:00:00,2019-04-01 00:00:00,...,2022-03-01 00:00:00,2022-04-01 00:00:00,2022-05-01 00:00:00,2022-06-01 00:00:00,2022-07-01 00:00:00,2022-08-01 00:00:00,2022-09-01 00:00:00,2022-10-01 00:00:00,2022-11-01 00:00:00,2022-12-01 00:00:00,2023-01-01 00:00:00,2023-02-01 00:00:00,2023-03-01 00:00:00,2023-04-01 00:00:00,2023-05-01 00:00:00,2023-06-01 00:00:00,2023-07-01 00:00:00,2023-08-01 00:00:00,2023-09-01 00:00:00,2023-10-01 00:00:00,2023-11-01 00:00:00,2023-12-01 00:00:00,2024-01-01 00:00:00,2024-02-01 00:00:00,2024-03-01 00:00:00,2024-04-01 00:00:00,2024-05-01 00:00:00,2024-06-01 00:00:00,2024-07-01 00:00:00,2024-08-01 00:00:00,2024-09-01 00:00:00,2024-10-01 00:00:00,2024-11-01 00:00:00,2024-12-01 00:00:00,2025-01-01 00:00:00,2025-02-01 00:00:00,2025-03-01 00:00:00,2025-04-01 00:00:00,2025-05-01 00:00:00,2025-06-01 00:00:00,2025-07-01 00:00:00,2025-08-01 00:00:00,2025-09-01 00:00:00,2025-10-01 00:00:00,2025-11-01 00:00:00,2025-12-01 00:00:00,2026-01-01 00:00:00,2026-02-01 00:00:00,2026-03-01 00:00:00,2026-04-01 00:00:00,2026-05-01 00:00:00,2026-06-01 00:00:00,18-1Q同営業日,18-2Q同営業日,18-3Q同営業日,18-4Q同営業日,19-1Q同営業日,19-2Q同営業日,19-3Q同営業日,19-4Q同営業日,20-1Q同営業日,20-2Q同営業日,20-3Q同営業日,20-4Q同営業日,21-1Q同営業日,21-2Q同営業日,21-3Q同営業日,21-4Q同営業日,22-1Q同営業日,22-2Q同営業日,22-3Q同営業日,22-4Q同営業日,23-1Q同営業日,23-2Q同営業日,23-3Q同営業日,23-4Q同営業日,24-1Q同営業日,24-2Q同営業日,24-3Q同営業日,24-4Q同営業日,25-1Q同営業日,25-2Q同営業日,25-3Q同営業日,25-4Q同営業日,26-1Q同営業日,26-2Q同営業日,26-3Q同営業日,26-4Q同営業日,27-1Q同営業日,27-2Q同営業日,27-3Q同営業日,27-4Q同営業日,28-1Q同営業日,28-2Q同営業日,28-3Q同営業日,28-4Q同営業日,29-1Q同営業日,29-2Q同営業日,29-3Q同営業日,nan同営業日
0,shoki_settei,APソース,SMAP,0.0,2.0,0.0,0.0,0.0,10.0,43.0,17.0,36.0,58.0,40.0,41.0,36.0,50.0,42.0,76.0,71.0,97.0,112.0,101.0,106.0,96.0,180.0,144.0,231.0,265.0,185.0,334.0,183.0,183.0,203.0,508.0,876.0,1173.0,928.0,930.0,1128.0,1192.0,1282.0,1414.0,1429.0,1426.0,1545.0,1568.0,1622.0,1577.0,994.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,2.0,2.0,19.0,9.0,16.0,9.0,7.0,1.0,8.0,20.0,8.0,17.0,18.0,22.0,16.0,14.0,10.0,15.0,14.0,12.0,12.0,18.0,6.0,16.0,14.0,20.0,3.0,7.0,32.0,30.0,25.0,20.0,22.0,32.0,18.0,27.0,29.0,43.0,32.0,...,74.0,64.0,69.0,70.0,115.0,160.0,230.0,273.0,320.0,297.0,258.0,458.0,467.0,250.0,357.0,311.0,276.0,337.0,290.0,366.0,398.0,402.0,398.0,401.0,398.0,383.0,454.0,439.0,508.0,350.0,532.0,490.0,494.0,470.0,490.0,438.0,516.0,508.0,529.0,467.0,628.0,428.0,510.0,504.0,540.0,597.0,467.0,494.0,597.0,487.0,414.0,93.0,0.0,2.0,0.0,0.0,0.0,10.0,43.0,17.0,36.0,58.0,40.0,41.0,36.0,50.0,42.0,76.0,71.0,97.0,112.0,101.0,106.0,96.0,180.0,144.0,231.0,265.0,185.0,334.0,183.0,183.0,203.0,508.0,876.0,1173.0,928.0,930.0,1128.0,1192.0,1282.0,1414.0,1429.0,1426.0,1545.0,1568.0,1622.0,1577.0,994.0,1.0
1,shoki_settei,APソース,その他,0.0,0.0,0.0,1.0,1.0,11.0,26.0,30.0,57.0,118.0,68.0

In [ ]:
# # ==========================================
# # 最終データの書き出し前の「究極の文字列化」処理
# # ==========================================

# # 1. 全データを文字型に変換（ガードマンを退場させる）
# export_output_df = output_df.astype(str)

# # 2. 「日付の空っぽ（NaT）」や「文字になったnan」を綺麗な空文字に直す
# export_output_df = export_output_df.replace(["nan", "NaN", "<NA>", "None", "NaT"], "")

# # 3. 列の名前（ヘッダー）も忘れずに文字型に変換
# export_output_df.columns = export_output_df.columns.astype(str)

# # 4. 送信用データ（リスト型）の作成
# values_to_send = [export_output_df.columns.tolist()] + export_output_df.values.tolist()

# # ==========================================
# # スプレッドシートへの書き出し
# # ==========================================
# print("最終データの書き出しを開始します...")
# ps.update_ss(RESULT_SPREADSHEET_ID, f"{SHEET_NAME}!B1", values_to_send, service)

# print("✅ 全データの統合とスプレッドシートへの出力が完了しました！")
# display(output_df.head())